<a href="https://colab.research.google.com/github/osergioribeirof/Python/blob/main/SR_GammaFlip__Interativo_V2_CBOE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### BIBLIOTECA

In [100]:
### Rodar essa célula somente uma vez ###
# Delete a # na linha abaixo, execute e coloque de volta a #

#!pip install plotly

In [101]:
import pandas as pd
import plotly
pd.set_option('plotting.backend','plotly')
import plotly.graph_objs as go
import numpy as np
import scipy
from scipy.stats import norm
#import matplotlib.pyplot as plt
import calendar
from datetime import datetime, timedelta, date

In [102]:
pd.options.display.float_format = '{:,.4f}'.format

### ARQUIVO CSV

In [103]:
# Parametros de entrada
filename = 'quotedata.csv'

# Black-Scholes European-Options Gamma
def calcGammaEx(S, K, vol, T, r, q, optType, OI):
    if T == 0 or vol == 0:
        return 0

    dp = (np.log(S/K) + (r - q + 0.5*vol**2)*T) / (vol*np.sqrt(T))
    dm = dp - vol*np.sqrt(T)

    if optType == 'call':
        gamma = np.exp(-q*T) * norm.pdf(dp) / (S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma
    else: # Gamma is same for calls and puts. This is just to cross-check
        gamma = K * np.exp(-r*T) * norm.pdf(dm) / (S * S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma

def isThirdFriday(d):
    return d.weekday() == 4 and 15 <= d.day <= 21

In [104]:
# Isso assume que o formato do arquivo CBOE não foi editado, ou seja, a tabela começa na linha 4
optionsFile = open(filename)
optionsFileData = optionsFile.readlines()
optionsFile.close()

In [105]:
# Extraindo SPX spot
spotLine = optionsFileData[1]
spotPrice = float(spotLine.split('Last:')[1].split(',')[0])
fromStrike = 0.8 * spotPrice
toStrike = 1.2 * spotPrice

In [106]:
# Extraindo a data de hoje
dateLine = optionsFileData[2]
todayDate = dateLine.split('Date: ')[1].split(',')
monthDay = todayDate[0].split(' ')

In [107]:
if len(monthDay) == 2:
    year = int(monthDay[4])
    month = monthDay[2]
    day = int(monthDay[0])
else:
    if monthDay[2].isdigit():
        year = int(monthDay[4])
        month = monthDay[2]
        day = int(monthDay[0])
    else:
        year = int(monthDay[4])
        month = monthDay[2]
        day = int(monthDay[0])


# criar um dicionário para mapear os nomes dos meses em português para os equivalentes em inglês
nomes_meses = {'janeiro': 'January', 'fevereiro': 'February', 'março': 'March',
               'abril': 'April', 'maio': 'May', 'junho': 'June',
               'julho': 'July', 'agosto': 'August', 'setembro': 'September',
               'outubro': 'October', 'novembro': 'November', 'dezembro': 'December'}

# extrair o nome do mês da string de entrada
nome_mes_pt = monthDay[2]

# converter o nome do mês para inglês usando o dicionário
nome_mes_en = nomes_meses[nome_mes_pt]

# converter o nome do mês para o número correspondente (por exemplo, 'March' -> 3)
num_mes = datetime.strptime(nome_mes_en, '%B').month

# criar o objeto datetime
todayDate = datetime(year=year, month=num_mes, day=day)

In [108]:
# create a dictionary to map Portuguese month names to English month names
month_names = {'janeiro': 'January', 'fevereiro': 'February', 'março': 'March',
               'abril': 'April', 'maio': 'May', 'junho': 'June',
               'julho': 'July', 'agosto': 'August', 'setembro': 'September',
               'outubro': 'October', 'novembro': 'November', 'dezembro': 'December'}

# extract the month name from the input string
month_name_pt = monthDay[2]

# convert the month name to English using the dictionary
month_name_en = month_names[month_name_pt]

# convert the month name to its corresponding number (e.g., 'March' -> 3)
month_number = datetime.strptime(month_name_en, '%B').month

# create the datetime object
todayDate = datetime(year=year, month=month_number, day=day)

In [109]:
# Get SPX Options Data
df = pd.read_csv(filename, sep=",", header=None, skiprows=4)
df.columns = ['ExpirationDate','Calls','CallLastSale','CallNet','CallBid','CallAsk','CallVol',
              'CallIV','CallDelta','CallGamma','CallOpenInt','StrikePrice','Puts','PutLastSale',
              'PutNet','PutBid','PutAsk','PutVol','PutIV','PutDelta','PutGamma','PutOpenInt']


df['ExpirationDate'] = pd.to_datetime(df['ExpirationDate'], format='%a %b %d %Y')
df['ExpirationDate'] = df['ExpirationDate'] + timedelta(hours=16)
df['StrikePrice'] = df['StrikePrice'].astype(float)
df['CallIV'] = df['CallIV'].astype(float)
df['PutIV'] = df['PutIV'].astype(float)
df['CallGamma'] = df['CallGamma'].astype(float)
df['PutGamma'] = df['PutGamma'].astype(float)
df['CallOpenInt'] = df['CallOpenInt'].astype(float)
df['PutOpenInt'] = df['PutOpenInt'].astype(float)

### GAMMA (CHART 1, 2 E 3)

In [110]:
# ---=== CALCULATE SPOT GAMMA ===---
# Gamma Exposure = Unit Gamma * Open Interest * Contract Size * Spot Price
# To further convert into 'per 1% move' quantity, multiply by 1% of spotPrice
df['CallGEX'] = df['CallGamma'] * df['CallOpenInt'] * 100 * spotPrice * spotPrice * 0.01
df['PutGEX'] = df['PutGamma'] * df['PutOpenInt'] * 100 * spotPrice * spotPrice * 0.01 * -1

df['TotalGamma'] = (df.CallGEX + df.PutGEX) / 10**9
dfAgg = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes = dfAgg.index.values

In [111]:
# Chart 1: Absolute Gamma Exposure
# define os dados
x_data = strikes
y_data = dfAgg['TotalGamma'].to_numpy()

# cria um gráfico de barras
fig = go.Figure(
    go.Bar(
        x=x_data,
        y=y_data,
        width=6,
        marker_color='rgb(26, 118, 255)',  # cor das barras
        marker_line_color='black',  # cor das linhas de contorno das barras
        marker_line_width=0.15,  # largura das linhas de contorno das barras
        name='Gamma Exposure'
    )
)

# adiciona uma linha vertical para o preço spot
fig.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data),
    x1=spotPrice,
    y1=max(y_data),
    line=dict(
        color='red',
        width=2,
        dash='dash'  # estilo da linha
    )
)

# define o layout do gráfico
fig.update_layout(
    title={
        'text': f"Total Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black',}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Gamma Exposure ($ billions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',  # cor de fundo do gráfico
    font=dict(family='Arial', size=12, color='black')
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1750,
    height=800
)


# mostra o gráfico
fig.show()

In [112]:
# CALL WALL E PUT WALL - ALL DTE
call_oi_wall_strike = df.loc[df['CallOpenInt'].idxmax()]['StrikePrice']
call_oi_wall_value = df['CallOpenInt'].max()

# Find the strike with the highest Put Open Interest and its value
put_oi_wall_strike = df.loc[df['PutOpenInt'].idxmax()]['StrikePrice']
put_oi_wall_value = df['PutOpenInt'].max()

# Find the strike with the highest Call Volume and its value
call_vol_wall_strike = df.loc[df['CallVol'].idxmax()]['StrikePrice']
call_vol_wall_value = df['CallVol'].max()

# Find the strike with the highest Put Volume and its value
put_vol_wall_strike = df.loc[df['PutVol'].idxmax()]['StrikePrice']
put_vol_wall_value = df['PutVol'].max()

# Find the top 5 strikes with the highest Call Open Interest
top5_call_oi = df.nlargest(5, 'CallOpenInt')[['StrikePrice', 'CallOpenInt']]

# Find the top 5 strikes with the highest Put Open Interest
top5_put_oi = df.nlargest(5, 'PutOpenInt')[['StrikePrice', 'PutOpenInt']]


print("--- Dados de Open Interest e Volume para Call/Put Walls ---")

print("\nParedes por Open Interest:")
print(f"  Call Wall (OI): {call_oi_wall_value:.0f} at strike {call_oi_wall_strike:.0f}")
print(f"  Put Wall (OI): {put_oi_wall_value:.0f} at strike {put_oi_wall_strike:.0f}")

print("\nParedes por Volume:")
print(f"  Call Wall (Vol): {call_vol_wall_value:.0f} at strike {call_vol_wall_strike:.0f}")
print(f"  Put Wall (Vol): {put_vol_wall_value:.0f} at strike {put_vol_wall_strike:.0f}")

print("\nTop 5 Calls por Open Interest:")
for index, row in top5_call_oi.iterrows():
    print(f"  Strike {row['StrikePrice']:.0f}: {row['CallOpenInt']:.0f}")

print("\nTop 5 Puts por Open Interest:")
for index, row in top5_put_oi.iterrows():
    print(f"  Strike {row['StrikePrice']:.0f}: {row['PutOpenInt']:.0f}")

--- Dados de Open Interest e Volume para Call/Put Walls ---

Paredes por Open Interest:
  Call Wall (OI): 363849 at strike 5000
  Put Wall (OI): 399274 at strike 5000

Paredes por Volume:
  Call Wall (Vol): 85424 at strike 6900
  Put Wall (Vol): 73442 at strike 6850

Top 5 Calls por Open Interest:
  Strike 5000: 363849
  Strike 6000: 241557
  Strike 4000: 200147
  Strike 5000: 192934
  Strike 6000: 190980

Top 5 Puts por Open Interest:
  Strike 5000: 399274
  Strike 6000: 264839
  Strike 4000: 225537
  Strike 6000: 206531
  Strike 5000: 205096


In [113]:
# DADOS DO CHART 1
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')

# Get the top 9 smallest gamma values (most negative)
smallest_gamma = dfAgg_sorted.head(9)
print("Top 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente):")
# Iterate through smallest_gamma in descending order of TotalGamma (already sorted ascending, so reverse)
for index, row in smallest_gamma.iloc[::-1].iterrows():
    # Get the corresponding Delta value from dfAgg_delta
    delta_value = dfAgg_delta.loc[index, 'TotalDelta'] if index in dfAgg_delta.index else 0
    print(f"  Strike {index:.0f}: Type: P, Delta: {delta_value:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

# Get the top 9 largest gamma values (most positive)
largest_gamma = dfAgg_sorted.tail(9)
print("\nTop 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente):")
# Iterate through largest_gamma in descending order of TotalGamma (already sorted ascending, so reverse)
for index, row in largest_gamma.iloc[::-1].iterrows():
    # Get the corresponding Delta value from dfAgg_delta
    delta_value = dfAgg_delta.loc[index, 'TotalDelta'] if index in dfAgg_delta.index else 0
    print(f"  Strike {index:.0f}: Type: C, Delta: {delta_value:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

# Add the Zero Gamma (Gamma Flip) point
print(f"\nZero Gamma (Gamma Flip): {zeroGamma:.0f}")

# Print the total gamma
print(f"\nTotal Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")


# --- Find the single largest strike by Volume and Open Interest (Overall) ---

print("\n--- Strikes com Maior Volume e Open Interest (Geral) ---")

# Largest Call Volume Strike
largest_call_vol_strike = df.loc[df['CallVol'].idxmax()]
print(f"\nMaior Volume de Call: {largest_call_vol_strike['CallVol']:.0f} at strike {largest_call_vol_strike['StrikePrice']:.0f}")

# Largest Put Volume Strike
largest_put_vol_strike = df.loc[df['PutVol'].idxmax()]
print(f"Maior Volume de Put: {largest_put_vol_strike['PutVol']:.0f} at strike {largest_put_vol_strike['StrikePrice']:.0f}")

# Largest Call Open Interest Strike
largest_call_oi_strike = df.loc[df['CallOpenInt'].idxmax()]
print(f"\nMaior Open Interest de Call: {largest_call_oi_strike['CallOpenInt']:.0f} at strike {largest_call_oi_strike['StrikePrice']:.0f}")

# Largest Put Open Interest Strike
largest_put_oi_strike = df.loc[df['PutOpenInt'].idxmax()]
print(f"Maior Open Interest de Put: {largest_put_oi_strike['PutOpenInt']:.0f} at strike {largest_put_oi_strike['StrikePrice']:.0f}")

Top 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente):
  Strike 6330: Type: P, Delta: -3696.6134 M, Gamma: -0.7685 Bn
  Strike 5500: Type: P, Delta: 24255.0242 M, Gamma: -0.8078 Bn
  Strike 6200: Type: P, Delta: 24414.3456 M, Gamma: -1.0164 Bn
  Strike 6000: Type: P, Delta: 497895.0896 M, Gamma: -1.0387 Bn
  Strike 6400: Type: P, Delta: 29434.3140 M, Gamma: -1.0533 Bn
  Strike 6300: Type: P, Delta: 27429.8740 M, Gamma: -1.1908 Bn
  Strike 6500: Type: P, Delta: 49077.6203 M, Gamma: -1.4016 Bn
  Strike 6895: Type: P, Delta: 373.4368 M, Gamma: -1.5057 Bn
  Strike 6890: Type: P, Delta: 2799.1580 M, Gamma: -2.7925 Bn

Top 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente):
  Strike 7000: Type: C, Delta: 44987.9279 M, Gamma: 10.9452 Bn
  Strike 6950: Type: C, Delta: 23233.3433 M, Gamma: 5.8570 Bn
  Strike 6850: Type: C, Delta: 32224.7127 M, Gamma: 4.4717 Bn
  Strike 7100: Type: C, Delta: 21420.2435 M, Gamma: 3.9613 Bn
  Strike 6900: Type: C, Delta: 35600.4417 M, Gamma: 3.7872 

In [114]:
# Chart 2: Absolute Gamma Exposure by Calls and Puts
fig = go.Figure()
fig.add_bar(x=strikes, y=dfAgg['CallGEX'].to_numpy() / 10**9, width=6, name="Call Gamma")
fig.add_bar(x=strikes, y=dfAgg['PutGEX'].to_numpy() / 10**9, width=6, name="Put Gamma")
fig.update_xaxes(range=[fromStrike, toStrike])
chartTitle = "Total Gamma: $" + str("{:.2f}".format(df['TotalGamma'].sum())) + " Bn per 1% SPX Move"
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))
fig.update_xaxes(title_text="Strike")
fig.update_yaxes(title_text="Spot Gamma Exposure ($ billions/1% move)")
fig.add_shape(dict(type="line", x0=spotPrice, y0=0, x1=spotPrice, y1=max(dfAgg['CallGEX'].to_numpy() / 10**9), line=dict(color="black", width=2), name="SPX Spot:" + str("{:,.0f}".format(spotPrice))))

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1750,
    height=800
)

fig.show()


In [115]:
# DADOS CHART 2
dfAgg['AbsoluteTotalGEX'] = dfAgg['CallGEX'].abs() + dfAgg['PutGEX'].abs()

# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure
dfAgg_sorted_gex = dfAgg.sort_values(by='AbsoluteTotalGEX', ascending=False)

# Get the top 6 strikes based on combined absolute GEX
gex_levels = dfAgg_sorted_gex.head(6)

print("Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):")
for i in range(len(gex_levels)):
    strike = gex_levels.index[i]
    call_gex = gex_levels.iloc[i]['CallGEX'] / 10**9
    put_gex = gex_levels.iloc[i]['PutGEX'] / 10**9
    print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")

Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):
  GEX Level 1: Strike 7000, Call GEX: 16.6733 Bn, Put GEX: -5.7281 Bn
  GEX Level 2: Strike 6000, Call GEX: 8.1595 Bn, Put GEX: -9.1982 Bn
  GEX Level 3: Strike 6900, Call GEX: 10.4918 Bn, Put GEX: -6.7047 Bn
  GEX Level 4: Strike 6800, Call GEX: 7.6095 Bn, Put GEX: -5.3291 Bn
  GEX Level 5: Strike 6700, Call GEX: 5.8691 Bn, Put GEX: -6.1893 Bn
  GEX Level 6: Strike 6750, Call GEX: 5.3903 Bn, Put GEX: -4.7375 Bn


In [116]:
# ---=== CALCULATE GAMMA PROFILE ===---
levels = np.linspace(fromStrike, toStrike, 60)

# For 0DTE options, I'm setting DTE = 1 day, otherwise they get excluded
df['daysTillExp'] = [1/262 if (np.busday_count(todayDate.date(), x.date())) == 0 \
                           else np.busday_count(todayDate.date(), x.date())/262 for x in df.ExpirationDate]

nextExpiry = df['ExpirationDate'].min()

df['IsThirdFriday'] = [isThirdFriday(x) for x in df.ExpirationDate]
thirdFridays = df.loc[df['IsThirdFriday'] == True]
nextMonthlyExp = thirdFridays['ExpirationDate'].min()

totalGamma = []
totalGammaExNext = []
totalGammaExFri = []

In [117]:
# For each spot level, calc gamma exposure at that point
for level in levels:
    df['callGammaEx'] = df.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['CallIV'],
                                                          row['daysTillExp'], 0, 0, "call", row['CallOpenInt']), axis = 1)

    df['putGammaEx'] = df.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['PutIV'],
                                                         row['daysTillExp'], 0, 0, "put", row['PutOpenInt']), axis = 1)

    totalGamma.append(df['callGammaEx'].sum() - df['putGammaEx'].sum())

    exNxt = df.loc[df['ExpirationDate'] != nextExpiry]
    totalGammaExNext.append(exNxt['callGammaEx'].sum() - exNxt['putGammaEx'].sum())

    exFri = df.loc[df['ExpirationDate'] != nextMonthlyExp]
    totalGammaExFri.append(exFri['callGammaEx'].sum() - exFri['putGammaEx'].sum())

totalGamma = np.array(totalGamma) / 10**9
totalGammaExNext = np.array(totalGammaExNext) / 10**9
totalGammaExFri = np.array(totalGammaExFri) / 10**9

In [118]:
# Find Gamma Flip Point
zeroCrossIdx = np.where(np.diff(np.sign(totalGamma)))[0]

negGamma = totalGamma[zeroCrossIdx]
posGamma = totalGamma[zeroCrossIdx+1]
negStrike = levels[zeroCrossIdx]
posStrike = levels[zeroCrossIdx+1]

zeroGamma = posStrike - ((posStrike - negStrike) * posGamma/(posGamma-negGamma))
zeroGamma = zeroGamma[0]

In [119]:
# Chart 3: Gamma Exposure Profile
fig = go.Figure()

fig.add_trace(go.Scatter(x=levels, y=totalGamma, mode='lines', name='All Expiries'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExNext, mode='lines', name='Ex-Next Expiry'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle = "Gamma Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig.update_layout(title=chartTitle, xaxis_title='Index Price', yaxis_title='Gamma Exposure ($ billions/1% move)')
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))

fig.add_shape(
    dict(
        type="line",
        x0=spotPrice,
        y0=min(totalGamma),
        x1=spotPrice,
        y1=max(totalGamma),
        line=dict(color="red", width=1.5),
        name="SPX Spot: " + str("{:,.0f}".format(spotPrice))
    )
)

fig.add_shape(
    dict(
        type="line",
        x0=zeroGamma,
        y0=min(totalGamma),
        x1=zeroGamma,
        y1=max(totalGamma),
        line=dict(color="green", width=1.5),
        name="Gamma Flip: " + str("{:,.0f}".format(zeroGamma))
    )
)

fig.update_xaxes(range=[fromStrike, toStrike])
fig.update_yaxes(range=[min(totalGamma), max(totalGamma)])

fig.add_trace(
    go.Scatter(
        x=[fromStrike, zeroGamma, toStrike],
        y=[min(totalGamma), min(totalGamma), min(totalGamma)],
        mode="none",
        fill="toself",
        fillcolor="red",
        opacity=0.1,
        showlegend=False,
        name="Negative Gamma"
    )
)

fig.add_trace(
    go.Scatter(
        x=[fromStrike, zeroGamma, toStrike],
        y=[max(totalGamma), max(totalGamma), max(totalGamma)],
        mode="none",
        fill="toself",
        fillcolor="green",
        opacity=0.1,
        showlegend=False,
        name="Positive Gamma"
    )
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1400,
    height=700
)

fig.show()

In [120]:
# DADOS CHART 3
# Gamma Flip (primeiro valor acima da linha verde central)
gamma_flip_index = np.where(totalGammaExFri > 0)[0][0] if np.any(totalGammaExFri > 0) else None
if gamma_flip_index is not None:
    gamma_flip_strike = levels[gamma_flip_index]
    gamma_flip_value = totalGammaExFri[gamma_flip_index]
    print(f"Gamma Flip (Ex-Next Monthly Expiry): {gamma_flip_value:.4f} at strike {gamma_flip_strike:.0f}")
else:
    print("No Gamma Flip found (Ex-Next Monthly Expiry).")

# Vol Trigger (value at zero gamma cross) for Ex-Next Monthly Expiry
vol_trigger_value_at_flip_exfri = np.interp(zeroGamma, levels, totalGammaExFri)
print(f"Vol Trigger (Gamma Flip Point, Ex-Next Monthly Expiry): {vol_trigger_value_at_flip_exfri:.4f} at strike {zeroGamma:.0f}")


# Max Gamma Positivo
max_gamma_positive_value = np.max(totalGammaExFri)
max_gamma_positive_index = np.argmax(totalGammaExFri)
max_gamma_positive_strike = levels[max_gamma_positive_index]
print(f"Max Gamma Positivo (Ex-Next Monthly Expiry): {max_gamma_positive_value:.4f} at strike {max_gamma_positive_strike:.0f}")


# Min Gamma Negativo
min_gamma_negative_value = np.min(totalGammaExFri)
min_gamma_negative_index = np.argmin(totalGammaExFri)
min_gamma_negative_strike = levels[min_gamma_negative_index]
print(f"Min Gamma Negativo (Ex-Next Monthly Expiry): {min_gamma_negative_value:.4f} at strike {min_gamma_negative_strike:.0f}")

Gamma Flip (Ex-Next Monthly Expiry): 12.1592 at strike 6821
Vol Trigger (Gamma Flip Point, Ex-Next Monthly Expiry): -3.0775 at strike 6773
Max Gamma Positivo (Ex-Next Monthly Expiry): 64.8422 at strike 7007
Min Gamma Negativo (Ex-Next Monthly Expiry): -65.0810 at strike 5886


### DELTA (CHART 4,5 E 6)

In [121]:
# ---=== CALCULATE SPOT DELTA ===---
# Delta Exposure = Unit Delta * Open Interest * Contract Size * Spot Price
df['CallDEX'] = df['CallDelta'] * df['CallOpenInt'] * 100 * spotPrice
df['PutDEX'] = df['PutDelta'] * df['PutOpenInt'] * 100 * spotPrice

# Total Delta considers the sign of delta for calls and puts
df['TotalDelta'] = (df.CallDEX + df.PutDEX) / 10**6 # Converting to millions for better scaling

dfAgg_delta = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes_delta = dfAgg_delta.index.values

# ---=== CALCULATE DELTA PROFILE ===---
levels_delta = np.linspace(fromStrike, toStrike, 60)

totalDelta = []
totalDeltaExNext = []
totalDeltaExFri = []

# For each spot level, calc delta exposure at that point
for level in levels_delta:
    df['callDeltaEx'] = df.apply(lambda row : row['CallDelta'] * row['CallOpenInt'] * 100 * level, axis = 1)

    df['putDeltaEx'] = df.apply(lambda row : row['PutDelta'] * row['PutOpenInt'] * 100 * level, axis = 1)

    totalDelta.append(df['callDeltaEx'].sum() + df['putDeltaEx'].sum())

    exNxt_delta = df.loc[df['ExpirationDate'] != nextExpiry]
    totalDeltaExNext.append(exNxt_delta['callDeltaEx'].sum() + exNxt_delta['putDeltaEx'].sum())

    exFri_delta = df.loc[df['ExpirationDate'] != nextMonthlyExp]
    totalDeltaExFri.append(exFri_delta['callDeltaEx'].sum() + exFri_delta['putDeltaEx'].sum())

totalDelta = np.array(totalDelta) / 10**6 # Converting to millions
totalDeltaExNext = np.array(totalDeltaExNext) / 10**6 # Converting to millions
totalDeltaExFri = np.array(totalDeltaExFri) / 10**6 # Converting to millions

# Find Delta Flip Point
zeroCrossIdx_delta = np.where(np.diff(np.sign(totalDelta)))[0]

# Handle the case where there is no zero cross
if zeroCrossIdx_delta.size > 0:
    negDelta = totalDelta[zeroCrossIdx_delta]
    posDelta = totalDelta[zeroCrossIdx_delta+1]
    negStrike_delta = levels_delta[zeroCrossIdx_delta]
    posStrike_delta = levels_delta[zeroCrossIdx_delta+1]

    zeroDelta = posStrike_delta - ((posStrike_delta - negStrike_delta) * posDelta/(posDelta-negDelta))
    # Keep zeroDelta as a single value if there's a cross, otherwise set to None or a default
    zeroDelta = zeroDelta[0] if zeroDelta.size > 0 else None
else:
    zeroDelta = None # Set zeroDelta to None if no zero cross is found

In [122]:
# Chart 4: Absolute Delta Exposure
# define os dados
x_data_delta = strikes_delta
y_data_delta = dfAgg_delta['TotalDelta'].to_numpy()

# cria um gráfico de barras
fig_delta4 = go.Figure(
    go.Bar(
        x=x_data_delta,
        y=y_data_delta,
        width=6,
        marker_color='rgb(26, 118, 255)',  # cor das barras
        marker_line_color='black',  # cor das linhas de contorno das barras
        marker_line_width=0.15,  # largura das linhas de contorno das barras
        name='Delta Exposure'
    )
)

# adiciona uma linha vertical para o preço spot
fig_delta4.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data_delta),
    x1=spotPrice,
    y1=max(y_data_delta),
    line=dict(
        color='red',
        width=2,
        dash='dash'  # estilo da linha
    )
)

# define o layout do gráfico
fig_delta4.update_layout(
    title={
        'text': f"Total Delta: ${df['TotalDelta'].sum():,.2f} Million per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black',}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Delta Exposure ($ millions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',  # cor de fundo do gráfico
    font=dict(family='Arial', size=12, color='black')
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig_delta4.update_layout(
    width=1750,
    height=800
)


# mostra o gráfico
fig_delta4.show()

In [123]:
# --- DADOS DO CHART 4 (Delta Exposure) ---
print("="*80)
print("📊 DADOS DO CHART 4 (Delta Exposure)")
print("="*80)

# Requires dfAgg_delta from cell 5f01c222
# Requires totalDelta from cell 6143f354
# Requires zeroDelta from cell 9e43625b
# Requires df from cell I3o4YVMQogB_

dfAgg_delta_sorted = dfAgg_delta.sort_values(by='TotalDelta')

# Get the top 9 smallest delta exposure values (most negative)
smallest_delta_exposure = dfAgg_delta_sorted.head(9)
print("\nTop 9 Strikes por Delta Exposure Negativo (Ordem Decrescente):")
for index, row in smallest_delta_exposure.iloc[::-1].iterrows():
    # Get the corresponding Gamma value from dfAgg
    gamma_value = dfAgg.loc[index, 'TotalGamma'] if index in dfAgg.index else 0
    print(f"  Strike {index:.0f}: Type: P, Gamma: {gamma_value:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")


# Get the top 9 largest delta exposure values (most positive)
largest_delta_exposure = dfAgg_delta_sorted.tail(9)
print("\nTop 9 Strikes por Delta Exposure Positivo (Ordem Decrescente):")
for index, row in largest_delta_exposure.iloc[::-1].iterrows():
    # Get the corresponding Gamma value from dfAgg
    gamma_value = dfAgg.loc[index, 'TotalGamma'] if index in dfAgg.index else 0
    print(f"  Strike {index:.0f}: Type: C, Gamma: {gamma_value:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")


# Add the Zero Delta (Delta Flip) point
# zeroDelta is calculated in cell 9e43625b
if zeroDelta is not None:
    print(f"\nZero Delta (Delta Flip): {zeroDelta:.0f}")
else:
    print("\nZero Delta (Delta Flip): Não encontrado")

# Print the total delta exposure
print(f"\nTotal Delta: ${df['TotalDelta'].sum():,.2f} Million per 1% SPX Move")


# --- Strikes com Maior Volume e Open Interest (Geral) ---
# Referencing variables already calculated in cell e8b1a91b

print("\n--- Strikes com Maior Volume e Open Interest (Geral) ---")

# Largest Call Volume Strike
# largest_call_vol_strike is from cell e8b1a91b
print(f"\nMaior Volume de Call: {largest_call_vol_strike['CallVol']:.0f} at strike {largest_call_vol_strike['StrikePrice']:.0f}")

# Largest Put Volume Strike
# largest_put_vol_strike is from cell e8b1a91b
print(f"Maior Volume de Put: {largest_put_vol_strike['PutVol']:.0f} at strike {largest_put_vol_strike['StrikePrice']:.0f}")

# Largest Call Open Interest Strike
# largest_call_oi_strike is from cell e8b1a91b
print(f"\nMaior Open Interest de Call: {largest_call_oi_strike['CallOpenInt']:.0f} at strike {largest_call_oi_strike['StrikePrice']:.0f}")

# Largest Put Open Interest Strike
# largest_put_oi_strike is from cell e8b1a91b
print(f"Maior Open Interest de Put: {largest_put_oi_strike['PutOpenInt']:.0f} at strike {largest_put_oi_strike['StrikePrice']:.0f}")

📊 DADOS DO CHART 4 (Delta Exposure)

Top 9 Strikes por Delta Exposure Negativo (Ordem Decrescente):
  Strike 11000: Type: P, Gamma: 0.0126 Bn, Delta: -308.6463 M
  Strike 4650: Type: P, Gamma: -0.0173 Bn, Delta: -311.7556 M
  Strike 4825: Type: P, Gamma: -0.0224 Bn, Delta: -339.4812 M
  Strike 6965: Type: P, Gamma: 0.0289 Bn, Delta: -391.2231 M
  Strike 2700: Type: P, Gamma: 0.0000 Bn, Delta: -406.3461 M
  Strike 6290: Type: P, Gamma: -0.1955 Bn, Delta: -500.8333 M
  Strike 5340: Type: P, Gamma: -0.1682 Bn, Delta: -606.2408 M
  Strike 12000: Type: P, Gamma: 0.0224 Bn, Delta: -1032.8980 M
  Strike 6330: Type: P, Gamma: -0.7685 Bn, Delta: -3696.6134 M

Top 9 Strikes por Delta Exposure Positivo (Ordem Decrescente):
  Strike 5000: Type: C, Gamma: -0.3636 Bn, Delta: 664797.8352 M
  Strike 6000: Type: C, Gamma: -1.0387 Bn, Delta: 497895.0896 M
  Strike 4000: Type: C, Gamma: 0.0000 Bn, Delta: 244326.1315 M
  Strike 6600: Type: C, Gamma: -0.3266 Bn, Delta: 55834.8764 M
  Strike 6700: Type: C, 

In [124]:
# Chart 5: Absolute Delta Exposure by Calls and Puts
fig_delta5 = go.Figure()
fig_delta5.add_bar(x=strikes_delta, y=dfAgg_delta['CallDEX'].to_numpy() / 10**6, width=6, name="Call Delta")
fig_delta5.add_bar(x=strikes_delta, y=dfAgg_delta['PutDEX'].to_numpy() / 10**6, width=6, name="Put Delta")
fig_delta5.update_xaxes(range=[fromStrike, toStrike])
chartTitle_delta5 = "Total Delta: $" + str("{:.2f}".format(df['TotalDelta'].sum())) + " Million per 1% SPX Move"
fig_delta5.update_layout(title_text=chartTitle_delta5, title_font=dict(size=20, family="Arial Black"))
fig_delta5.update_xaxes(title_text="Strike")
fig_delta5.update_yaxes(title_text="Spot Delta Exposure ($ millions/1% move)")
fig_delta5.add_shape(dict(type="line", x0=spotPrice, y0=min(dfAgg_delta['PutDEX'].to_numpy() / 10**6), x1=spotPrice, y1=max(dfAgg_delta['CallDEX'].to_numpy() / 10**6), line=dict(color="black", width=2), name="SPX Spot:" + str("{:,.0f}".format(spotPrice))))

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig_delta5.update_layout(
    width=1750,
    height=800
)

fig_delta5.show()

In [125]:
# ---=== CALCULATE DELTA PROFILE ===---
levels_delta = np.linspace(fromStrike, toStrike, 60)

totalDelta = []
totalDeltaExNext = []
totalDeltaExFri = []

# For each spot level, calc delta exposure at that point
for level in levels_delta:
    df['callDeltaEx'] = df.apply(lambda row : row['CallDelta'] * row['CallOpenInt'] * 100 * level, axis = 1)

    df['putDeltaEx'] = df.apply(lambda row : row['PutDelta'] * row['PutOpenInt'] * 100 * level, axis = 1)

    totalDelta.append(df['callDeltaEx'].sum() + df['putDeltaEx'].sum())

    exNxt_delta = df.loc[df['ExpirationDate'] != nextExpiry]
    totalDeltaExNext.append(exNxt_delta['callDeltaEx'].sum() + exNxt_delta['putDeltaEx'].sum())

    exFri_delta = df.loc[df['ExpirationDate'] != nextMonthlyExp]
    totalDeltaExFri.append(exFri_delta['callDeltaEx'].sum() + exFri_delta['putDeltaEx'].sum())

totalDelta = np.array(totalDelta) / 10**6 # Converting to millions
totalDeltaExNext = np.array(totalDeltaExNext) / 10**6 # Converting to millions
totalDeltaExFri = np.array(totalDeltaExFri) / 10**6 # Converting to millions

In [126]:
# Find Delta Flip Point
zeroCrossIdx_delta = np.where(np.diff(np.sign(totalDelta)))[0]

# Handle the case where there is no zero cross
if zeroCrossIdx_delta.size > 0:
    negDelta = totalDelta[zeroCrossIdx_delta]
    posDelta = totalDelta[zeroCrossIdx_delta+1]
    negStrike_delta = levels_delta[zeroCrossIdx_delta]
    posStrike_delta = levels_delta[zeroCrossIdx_delta+1]

    zeroDelta = posStrike_delta - ((posStrike_delta - negStrike_delta) * posDelta/(posDelta-negDelta))
    # Keep zeroDelta as a single value if there's a cross, otherwise set to None or a default
    zeroDelta = zeroDelta[0] if zeroDelta.size > 0 else None
else:
    zeroDelta = None # Set zeroDelta to None if no zero cross is found

In [127]:
# Chart 6: Delta Exposure Profile
fig_delta6 = go.Figure()

fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDelta, mode='lines', name='All Expiries'))
fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDeltaExNext, mode='lines', name='Ex-Next Expiry'))
fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDeltaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle_delta6 = "Delta Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig_delta6.update_layout(title=chartTitle_delta6, xaxis_title='Index Price', yaxis_title='Delta Exposure ($ millions/1% move)')
fig_delta6.update_layout(title_text=chartTitle_delta6, title_font=dict(size=20, family="Arial Black"))

fig_delta6.add_shape(
    dict(
        type="line",
        x0=spotPrice,
        y0=min(totalDelta),
        x1=spotPrice,
        y1=max(totalDelta),
        line=dict(color="red", width=1.5),
        name="SPX Spot: " + str("{:,.0f}".format(spotPrice))
    )
)

# Add the Delta Flip line only if zeroDelta is not None
if zeroDelta is not None:
    fig_delta6.add_shape(
        dict(
            type="line",
            x0=zeroDelta,
            y0=min(totalDelta),
            x1=zeroDelta,
            y1=max(totalDelta),
            line=dict(color="green", width=1.5),
            # Format zeroDelta as a scalar
            name="Delta Flip: " + str("{:,.0f}".format(float(zeroDelta)))
        )
    )


fig_delta6.update_xaxes(range=[fromStrike, toStrike])
fig_delta6.update_yaxes(range=[min(totalDelta), max(totalDelta)])

# Adding shaded areas for positive and negative delta
# Adjust shaded areas to account for potential None zeroDelta
if zeroDelta is not None:
    fig_delta6.add_trace(
        go.Scatter(
            x=[fromStrike, zeroDelta, toStrike],
            y=[min(totalDelta), min(totalDelta), min(totalDelta)],
            mode="none",
            fill="toself",
            fillcolor="red",
            opacity=0.1,
            showlegend=False,
            name="Negative Delta"
        )
    )

    fig_delta6.add_trace(
        go.Scatter(
            x=[fromStrike, zeroDelta, toStrike],
            y=[max(totalDelta), max(totalDelta), max(totalDelta)],
            mode="none",
            fill="toself",
            fillcolor="green",
            opacity=0.1,
            showlegend=False,
            name="Positive Delta"
        )
    )
else:
     # If no zeroDelta, the entire range is either positive or negative
    fill_color = 'green' if totalDelta[0] >= 0 else 'red'
    fig_delta6.add_trace(
        go.Scatter(
            x=[fromStrike, toStrike, toStrike, fromStrike],
            y=[min(totalDelta), min(totalDelta), max(totalDelta), max(totalDelta)],
            mode="none",
            fill="toself",
            fillcolor=fill_color,
            opacity=0.1,
            showlegend=False,
            name="Delta Region"
        )
    )


# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig_delta6.update_layout(
    width=1400,
    height=700
)

fig_delta6.show()

In [128]:
# Consolidating results from CHART 1, CHART 2, and CHART 3 - ALL DTE



print("="*80)
print("📊 RESUMO CONSOLIDADO DOS DADOS")
print("="*80)

# --- DADOS DO CHART 1 ---
print("\n--- DADOS DO CHART 1 (Gamma Exposure) ---")
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')

# Get the top 9 smallest gamma values (most negative)
smallest_gamma = dfAgg_sorted.head(9)
print("Top 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente):")
for index, row in smallest_gamma.iloc[::-1].iterrows():
    # Get the corresponding Delta value from dfAgg_delta
    delta_value = dfAgg_delta.loc[index, 'TotalDelta'] if index in dfAgg_delta.index else 0
    print(f"  Strike {index:.0f}: Type: P, Delta: {delta_value:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

# Get the top 9 largest gamma values (most positive)
largest_gamma = dfAgg_sorted.tail(9)
print("\nTop 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente):")
for index, row in largest_gamma.iloc[::-1].iterrows():
    # Get the corresponding Delta value from dfAgg_delta
    delta_value = dfAgg_delta.loc[index, 'TotalDelta'] if index in dfAgg_delta.index else 0
    print(f"  Strike {index:.0f}: Type: C, Delta: {delta_value:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

# Add the Zero Gamma (Gamma Flip) point
print(f"\nZero Gamma (Gamma Flip): {zeroGamma:.0f}")

# Print the total gamma
print(f"\nTotal Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")

# --- Dados de Open Interest e Volume para Call/Put Walls (from cell e8b1a91b) ---
print("\n--- Dados de Open Interest e Volume ---")

# Find the strike with the highest Call Open Interest and its value
call_oi_wall_strike = df.loc[df['CallOpenInt'].idxmax()]['StrikePrice']
call_oi_wall_value = df['CallOpenInt'].max()

# Find the strike with the highest Put Open Interest and its value
put_oi_wall_strike = df.loc[df['PutOpenInt'].idxmax()]['StrikePrice']
put_oi_wall_value = df['PutOpenInt'].max()

# Find the strike with the highest Call Volume and its value
call_vol_wall_strike = df.loc[df['CallVol'].idxmax()]['StrikePrice']
call_vol_wall_value = df['CallVol'].max()

# Find the strike with the highest Put Volume and its value
put_vol_wall_strike = df.loc[df['PutVol'].idxmax()]['StrikePrice']
put_vol_wall_value = df['PutVol'].max()

# Find the top 5 strikes with the highest Call Open Interest
top5_call_oi = df.nlargest(5, 'CallOpenInt')[['StrikePrice', 'CallOpenInt']]

# Find the top 5 strikes with the highest Put Open Interest
top5_put_oi = df.nlargest(5, 'PutOpenInt')[['StrikePrice', 'PutOpenInt']]

print("\nParedes por Open Interest:")
print(f"  Call Wall (OI): {call_oi_wall_value:.0f} at strike {call_oi_wall_strike:.0f}")
print(f"  Put Wall (OI): {put_oi_wall_value:.0f} at strike {put_oi_wall_strike:.0f}")

print("\nParedes por Volume:")
print(f"  Call Wall (Vol): {call_vol_wall_value:.0f} at strike {call_vol_wall_strike:.0f}")
print(f"  Put Wall (Vol): {put_vol_wall_value:.0f} at strike {put_vol_wall_strike:.0f}")

print("\nTop 5 Calls por Open Interest:")
for index, row in top5_call_oi.iterrows():
    print(f"  Strike {row['StrikePrice']:.0f}: {row['CallOpenInt']:.0f}")

print("\nTop 5 Puts por Open Interest:")
for index, row in top5_put_oi.iterrows():
    print(f"  Strike {row['StrikePrice']:.0f}: {row['PutOpenInt']:.0f}")

# --- DADOS CHART 2 (GEX Levels) ---
print("\n--- DADOS CHART 2 (GEX Levels) ---")
dfAgg['AbsoluteTotalGEX'] = dfAgg['CallGEX'].abs() + dfAgg['PutGEX'].abs()

# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure
dfAgg_sorted_gex = dfAgg.sort_values(by='AbsoluteTotalGEX', ascending=False)

# Get the top 6 strikes based on combined absolute GEX
gex_levels = dfAgg_sorted_gex.head(6)

print("Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):")
for i in range(len(gex_levels)):
    strike = gex_levels.index[i]
    call_gex = gex_levels.iloc[i]['CallGEX'] / 10**9
    put_gex = gex_levels.iloc[i]['PutGEX'] / 10**9
    print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")

# --- DADOS CHART 3 (Gamma Profile and Delta Flip) ---
print("\n--- DADOS CHART 3 (Gamma Profile & Delta Flip) ---")
# Gamma Flip (based on zeroGamma)
gamma_flip_index = np.where(totalGammaExFri > 0)[0][0] if np.any(totalGammaExFri > 0) else None
if gamma_flip_index is not None:
    gamma_flip_strike = levels[gamma_flip_index]
    gamma_flip_value = totalGammaExFri[gamma_flip_index]
    print(f"Gamma Flip (Ex-Next Monthly Expiry): {gamma_flip_value:.4f} at strike {gamma_flip_strike:.0f}")
else:
    print("No Gamma Flip found (Ex-Next Monthly Expiry).")

# Vol Trigger (value at zero gamma cross) for Ex-Next Monthly Expiry (based on zeroGamma)
vol_trigger_value_at_flip_exfri = np.interp(zeroGamma, levels, totalGammaExFri)
print(f"Vol Trigger (Gamma Flip Point, Ex-Next Monthly Expiry): {vol_trigger_value_at_flip_exfri:.4f} at strike {zeroGamma:.0f}")

# Max Gamma Positivo (Ex-Next Monthly Expiry)
max_gamma_positive_value = np.max(totalGammaExFri)
max_gamma_positive_index = np.argmax(totalGammaExFri)
max_gamma_positive_strike = levels[max_gamma_positive_index]
print(f"Max Gamma Positivo (Ex-Next Monthly Expiry): {max_gamma_positive_value:.4f} at strike {max_gamma_positive_strike:.0f}")

# Min Gamma Negativo (Ex-Next Monthly Expiry)
min_gamma_negative_value = np.min(totalGammaExFri) # Using totalDeltaExFri by mistake? Should be totalGammaExFri
min_gamma_negative_index = np.argmin(totalGammaExFri)
min_gamma_negative_strike = levels[min_gamma_negative_index] # Corrected variable name to min_gamma_negative_strike
print(f"Min Gamma Negativo (Ex-Next Monthly Expiry): {min_gamma_negative_value:.4f} at strike {min_gamma_negative_strike:.0f}")

# Delta Flip (Delta at Spot Price for Ex-Next Monthly Expiry line)
delta_at_spot_exfri = np.interp(spotPrice, levels_delta, totalDeltaExFri)
print(f"Delta Flip (Ex-Next Monthly Expiry): {delta_at_spot_exfri:.4f} Million at strike {spotPrice:.0f}")

📊 RESUMO CONSOLIDADO DOS DADOS

--- DADOS DO CHART 1 (Gamma Exposure) ---
Top 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente):
  Strike 6330: Type: P, Delta: -3696.6134 M, Gamma: -0.7685 Bn
  Strike 5500: Type: P, Delta: 24255.0242 M, Gamma: -0.8078 Bn
  Strike 6200: Type: P, Delta: 24414.3456 M, Gamma: -1.0164 Bn
  Strike 6000: Type: P, Delta: 497895.0896 M, Gamma: -1.0387 Bn
  Strike 6400: Type: P, Delta: 29434.3140 M, Gamma: -1.0533 Bn
  Strike 6300: Type: P, Delta: 27429.8740 M, Gamma: -1.1908 Bn
  Strike 6500: Type: P, Delta: 49077.6203 M, Gamma: -1.4016 Bn
  Strike 6895: Type: P, Delta: 373.4368 M, Gamma: -1.5057 Bn
  Strike 6890: Type: P, Delta: 2799.1580 M, Gamma: -2.7925 Bn

Top 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente):
  Strike 7000: Type: C, Delta: 44987.9279 M, Gamma: 10.9452 Bn
  Strike 6950: Type: C, Delta: 23233.3433 M, Gamma: 5.8570 Bn
  Strike 6850: Type: C, Delta: 32224.7127 M, Gamma: 4.4717 Bn
  Strike 7100: Type: C, Delta: 21420.2435 M, Ga

In [129]:
# Consolidating results from CHART 4, CHART 5, and CHART 6
print("--- DADOS CHART 4 ---")
# Requires dfAgg_delta from cell 5f01c222
# Requires df from cell I3o4YVMQogB_
# Requires zeroDelta from cell 9e43625b
# Requires largest_call_vol_strike, largest_put_vol_strike, largest_call_oi_strike, largest_put_oi_strike from cell e8b1a91b


dfAgg_delta_sorted = dfAgg_delta.sort_values(by='TotalDelta')

# Get the top 9 smallest delta exposure values (most negative)
smallest_delta_exposure = dfAgg_delta_sorted.head(9)
print("Top 9 Strikes por Delta Exposure Negativo (Ordem Decrescente):")
for index, row in smallest_delta_exposure.iloc[::-1].iterrows():
    # Get the corresponding Gamma value from dfAgg
    gamma_value = dfAgg.loc[index, 'TotalGamma'] if index in dfAgg.index else 0
    print(f"  Strike {index:.0f}: Type: P, Gamma: {gamma_value:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")


# Get the top 9 largest delta exposure values (most positive)
largest_delta_exposure = dfAgg_delta_sorted.tail(9)
print("\nTop 9 Strikes por Delta Exposure Positivo (Ordem Decrescente):")
for index, row in largest_delta_exposure.iloc[::-1].iterrows():
    # Get the corresponding Gamma value from dfAgg
    gamma_value = dfAgg.loc[index, 'TotalGamma'] if index in dfAgg.index else 0
    print(f"  Strike {index:.0f}: Type: C, Gamma: {gamma_value:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")


# Add the Zero Delta (Delta Flip) point (based on zero cross)
if zeroDelta is not None:
    print(f"\nZero Delta (Delta Flip - Zero Cross): {zeroDelta:.0f}")
else:
    print("\nZero Delta (Delta Flip - Zero Cross): Não encontrado")


# Print the total delta exposure
print(f"\nTotal Delta: ${df['TotalDelta'].sum():,.2f} Million per 1% SPX Move")

# --- Strikes com Maior Volume e Open Interest (Geral) ---
# Referencing variables already calculated in cell e8b1a91b

print("\n--- Strikes com Maior Volume e Open Interest (Geral) ---")

# Largest Call Volume Strike
# largest_call_vol_strike is from cell e8b1a91b
print(f"\nMaior Volume de Call: {largest_call_vol_strike['CallVol']:.0f} at strike {largest_call_vol_strike['StrikePrice']:.0f}")

# Largest Put Volume Strike
# largest_put_vol_strike is from cell e8b1a91b
print(f"Maior Volume de Put: {largest_put_vol_strike['PutVol']:.0f} at strike {largest_put_vol_strike['StrikePrice']:.0f}")

# Largest Call Open Interest Strike
# largest_call_oi_strike is from cell e8b1a91b
print(f"\nMaior Open Interest de Call: {largest_call_oi_strike['CallOpenInt']:.0f} at strike {largest_call_oi_strike['StrikePrice']:.0f}")

# Largest Put Open Interest Strike
# largest_put_oi_strike is from cell e8b1a91b
print(f"Maior Open Interest de Put: {largest_put_oi_strike['PutOpenInt']:.0f} at strike {largest_put_oi_strike['StrikePrice']:.0f}")


print("\n--- DADOS CHART 5 ---")
# DATA FOR CHART 5 (Absolute Delta Exposure by Calls and Puts)
# Calculate the absolute sum of Call and Put Delta Exposure for each strike
dfAgg_delta['AbsoluteTotalDEX'] = dfAgg_delta['CallDEX'].abs() + dfAgg_delta['PutDEX'].abs()

# Sort by AbsoluteTotalDEX to find the strikes with the largest combined exposure
dfAgg_delta_sorted_dex = dfAgg_delta.sort_values(by='AbsoluteTotalDEX', ascending=False)

# Get the top 6 strikes based on combined absolute DEX
dex_levels = dfAgg_delta_sorted_dex.head(6)

print("Top 6 DEX Levels (based on sum of absolute Call and Put Delta Exposure):")
for i in range(len(dex_levels)):
    strike_dex = dex_levels.index[i]
    call_dex = dex_levels.iloc[i]['CallDEX'] / 10**6
    put_dex = dex_levels.iloc[i]['PutDEX'] / 10**6
    print(f"  DEX Level {i+1}: Strike {strike_dex:.0f}, Call DEX: {call_dex:.4f} Million, Put DEX: {put_dex:.4f} Million")

print("\n--- DADOS CHART 6 ---")
# DATA FOR CHART 6 (Delta Exposure Profile)
# Find Delta Flip (first value above the central green line) for Ex-Next Monthly Expiry Delta Profile
# This assumes totalDeltaExFri is the data for the 'Ex-Next Monthly Expiry' line
# delta_flip_index_exfri = np.where(totalDeltaExFri > 0)[0][0] if np.any(totalDeltaExFri > 0) else None
# if delta_flip_index_exfri is not None:
#     delta_flip_strike_exfri = levels_delta[delta_flip_index_exfri]
#     delta_flip_value_exfri = totalDeltaExFri[delta_flip_index_exfri]
#     print(f"Delta Flip (Ex-Next Monthly Expiry): {delta_flip_value_exfri:.4f} at strike {delta_flip_strike_exfri:.0f}")
# else:
#     print("No Delta Flip found (Ex-Next Monthly Expiry).")

# Delta Vol Trigger (value at zero delta cross) for Ex-Next Monthly Expiry
# Use the zeroDelta calculated earlier for the Delta Flip point
# if zeroDelta is not None:
#     delta_vol_trigger_value_at_flip_exfri = np.interp(zeroDelta, levels_delta, totalDeltaExFri)
#     print(f"Delta Vol Trigger (Delta Flip Point, Ex-Next Monthly Expiry): {delta_vol_trigger_value_at_flip_exfri:.4f} at strike {zeroDelta:.0f}")
# else:
#     print("Delta Vol Trigger not found (no Delta Flip point).")

# Max Positive Delta (Ex-Next Monthly Expiry)
max_delta_positive_value_exfri = np.max(totalDeltaExFri)
max_delta_positive_index_exfri = np.argmax(totalDeltaExFri)
max_delta_positive_strike_exfri = levels_delta[max_delta_positive_index_exfri]
print(f"Max Delta Positivo (Ex-Next Monthly Expiry): {max_delta_positive_value_exfri:.4f} at strike {max_delta_positive_strike_exfri:.0f}")

# Min Negative Delta (Ex-Next Monthly Expiry)
min_delta_negative_value_exfri = np.min(totalDeltaExFri)
min_delta_negative_index_exfri = np.argmin(totalDeltaExFri)
min_delta_negative_strike_exfri = levels_delta[min_delta_negative_index_exfri]
print(f"Min Delta Negativo (Ex-Next Monthly Expiry): {min_delta_negative_value_exfri:.4f} at strike {min_delta_negative_strike_exfri:.0f}")

# Find the Delta Exposure value at the Spot Price for the Ex-Next Monthly Expiry line and label as Delta Flip
# delta_at_spot_exfri is already calculated above
print(f"Delta Flip (Ex-Next Monthly Expiry): {delta_at_spot_exfri:.4f} Million at strike {spotPrice:.0f}")

--- DADOS CHART 4 ---
Top 9 Strikes por Delta Exposure Negativo (Ordem Decrescente):
  Strike 11000: Type: P, Gamma: 0.0126 Bn, Delta: -308.6463 M
  Strike 4650: Type: P, Gamma: -0.0173 Bn, Delta: -311.7556 M
  Strike 4825: Type: P, Gamma: -0.0224 Bn, Delta: -339.4812 M
  Strike 6965: Type: P, Gamma: 0.0289 Bn, Delta: -391.2231 M
  Strike 2700: Type: P, Gamma: 0.0000 Bn, Delta: -406.3461 M
  Strike 6290: Type: P, Gamma: -0.1955 Bn, Delta: -500.8333 M
  Strike 5340: Type: P, Gamma: -0.1682 Bn, Delta: -606.2408 M
  Strike 12000: Type: P, Gamma: 0.0224 Bn, Delta: -1032.8980 M
  Strike 6330: Type: P, Gamma: -0.7685 Bn, Delta: -3696.6134 M

Top 9 Strikes por Delta Exposure Positivo (Ordem Decrescente):
  Strike 5000: Type: C, Gamma: -0.3636 Bn, Delta: 664797.8352 M
  Strike 6000: Type: C, Gamma: -1.0387 Bn, Delta: 497895.0896 M
  Strike 4000: Type: C, Gamma: 0.0000 Bn, Delta: 244326.1315 M
  Strike 6600: Type: C, Gamma: -0.3266 Bn, Delta: 55834.8764 M
  Strike 6700: Type: C, Gamma: -0.3202 

### ARQUIVO PARA TRADING VIEW

In [130]:
# ==================== CÉLULA SIMPLIFICADA PARA TRADING VIEW ====================
# Esta célula gera uma string de dados reduzida para o TradingView

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA/DELTA FLIP (VERSÃO SIMPLIFICADA)")
print("="*80 + "\n")

# ==================== COLETA DOS DADOS (SIMPLIFICADA) ====================
# Referencing variables calculated in preceding cells

# Dados gerais
spot_price = spotPrice # from fvEJmuujccxO
total_gamma = df['TotalGamma'].sum() # from I3o4YVMQogB_
# total_delta = df['TotalDelta'].sum() # Removed as per user request
update_date = todayDate.strftime('%d %b %Y 00:00') # from fvEJmuujccxO

# CHART 1 - Spot Gamma Levels (Simplified)
# Using the sorted dataframes to get the top/bottom strikes by gamma exposure
dfAgg_sorted_gamma = dfAgg.sort_values(by='TotalGamma')

# Top 5 Strikes por Gamma Exposure Negativo (os maiores - should be smallest/most negative)
# User asked for "os maiores" which is ambiguous for negative numbers. Assuming they mean the 5 most negative.
smallest_gamma_5 = dfAgg_sorted_gamma.head(5)
chart1_neg_gamma_data_simple = []
for index, row in smallest_gamma_5.iloc[::-1].iterrows(): # Iterate in descending order of gamma value
    chart1_neg_gamma_data_simple.append({'strike': index, 'gamma': row['TotalGamma']})


# Top 5 Strikes por Gamma Exposure Positivo (os maiores)
largest_gamma_5 = dfAgg_sorted_gamma.tail(5)
chart1_pos_gamma_data_simple = []
for index, row in largest_gamma_5.iloc[::-1].iterrows(): # Iterate in descending order of gamma value
     chart1_pos_gamma_data_simple.append({'strike': index, 'gamma': row['TotalGamma']})


# Zero Gamma (Gamma Flip) point (from REAlEsxDtvng and be13ea2c)
# zeroGamma is from REAlEsxDtvng and be13ea2c
chart1_zero_gamma_strike = zeroGamma


# Open Interest and Volume Walls (from e8b1a91b and be13ea2c)
# Using the variables already calculated in e8b1a91b and be13ea2c
chart1_call_oi_wall_strike = call_oi_wall_strike
chart1_call_oi_wall_value = call_oi_wall_value

chart1_put_oi_wall_strike = put_oi_wall_strike
chart1_put_oi_wall_value = put_oi_wall_value

chart1_call_vol_wall_strike = call_vol_wall_strike
chart1_call_vol_wall_value = call_vol_wall_value
chart1_put_vol_wall_strike = put_vol_wall_strike
chart1_put_vol_wall_value = put_vol_wall_value

# Top 5 Calls por Open Interest (from e8b1a91b and be13ea2c)
# top5_call_oi is from e8b1a91b and be13ea2c
chart1_top5_call_oi_data = top5_call_oi.to_dict('records')

# Top 5 Puts por Open Interest (from e8b1a91b and be13ea2c)
# top5_put_oi is from e8b1a91b and be13ea2c
chart1_top5_put_oi_data = top5_put_oi.to_dict('records')

# Strikes com Maior Volume e Open Interest (Geral) (from e8b1a91b and be13ea2c)
# largest_call_vol_strike, largest_put_vol_strike, largest_call_oi_strike, largest_put_oi_strike are from e8b1a91b and be13ea2c
chart1_largest_call_vol_strike = largest_call_vol_strike['StrikePrice']
chart1_largest_call_vol_value = largest_call_vol_strike['CallVol']
chart1_largest_put_vol_strike = largest_put_vol_strike['StrikePrice']
chart1_largest_put_vol_value = largest_put_vol_strike['PutVol']
chart1_largest_call_oi_strike = largest_call_oi_strike['StrikePrice']
chart1_largest_call_oi_value = largest_call_oi_strike['CallOpenInt']
# User requested "maior open interest" twice, assuming they meant Call and Put
chart1_largest_put_oi_strike = largest_put_oi_strike['StrikePrice']
chart1_largest_put_oi_value = largest_put_oi_strike['PutOpenInt']


# CHART 2 - GEX Levels (Simplified)
# gex_levels is from d0765a4d-2f3d-449c-a61b-9ef63ffb85ac and be13ea2c
# User requested Top 5 GEX levels
chart2_gex_levels_data_simple = []
for i in range(min(5, len(gex_levels))): # Take min of 5 or available levels
    chart2_gex_levels_data_simple.append({
        'strike': gex_levels.index[i],
        'call_gex': gex_levels.iloc[i]['CallGEX'] / 10**9, # Convert to billions
        'put_gex': gex_levels.iloc[i]['PutGEX'] / 10**9   # Convert to billions
    })


# CHART 3 - Gamma Profile points (from 0eb2b7f4 and be13ea2c)
# gamma_flip_value, gamma_flip_strike, max_gamma_positive_value, max_gamma_positive_strike,
# min_gamma_negative_value, min_gamma_negative_strike are from 0eb2b7f4 and be13ea2c
chart3_gamma_flip_strike = gamma_flip_strike
chart3_gamma_flip_value = gamma_flip_value
chart3_vol_trigger_strike = zeroGamma # Vol Trigger strike is the Gamma Flip point
chart3_vol_trigger_value = vol_trigger_value_at_flip_exfri # Vol Trigger value at Gamma Flip point
chart3_max_pos_strike = max_gamma_positive_strike
chart3_max_pos_value = max_gamma_positive_value
chart3_min_neg_strike = min_gamma_negative_strike
chart3_min_neg_value = min_gamma_negative_value

# Delta Flip (Estrutura) from Chart 3 (based on Delta at Spot Price for Ex-Next Monthly Expiry line)
chart3_delta_flip_estrutura_strike = spotPrice # Delta Flip (Estrutura) strike is the Spot Price
chart3_delta_flip_estrutura_value = np.interp(spotPrice, levels_delta, totalDeltaExFri) # Interpolate Delta value at Spot Price


# CHART 4 - Delta Exposure points (Simplified)
# dfAgg_delta_sorted is from 035f55f4
dfAgg_delta_sorted = dfAgg_delta.sort_values(by='TotalDelta')

# Top 5 Strikes por Delta Exposure Negativo (os maiores - should be smallest/most negative)
# User asked for "os maiores" which is ambiguous for negative numbers. Assuming they mean the 5 most negative.
smallest_delta_exposure_5 = dfAgg_delta_sorted.head(5)
chart4_neg_delta_data_simple = []
for index, row in smallest_delta_exposure_5.iloc[::-1].iterrows(): # Iterate in descending order of delta value
    chart4_neg_delta_data_simple.append({'strike': index, 'delta': row['TotalDelta']})


# Top 5 Strikes por Delta Exposure Positivo (os maiores)
# User requested Top 9 here, changing to Top 5 as requested in the latest turn.
largest_delta_exposure_5 = dfAgg_delta_sorted.tail(5)
chart4_pos_delta_data_simple = []
for index, row in largest_delta_exposure_5.iloc[::-1].iterrows(): # Iterate in descending order of delta value
    chart4_pos_delta_data_simple.append({'strike': index, 'delta': row['TotalDelta']})


# CHART 5 - DEX Levels (Simplified)
# dex_levels is from 035f55f4
# User requested Top 5 DEX levels
chart5_dex_levels_data_simple = []
for i in range(min(5, len(dex_levels))): # Take min of 5 or available levels
    chart5_dex_levels_data_simple.append({
        'strike': dex_levels.index[i],
        'call_dex': dex_levels.iloc[i]['CallDEX'] / 10**6, # Convert to Millions
        'put_dex': dex_levels.iloc[i]['PutDEX'] / 10**6   # Convert to Millions
    })


# CHART 6 - Delta Profile points (from 035f55f4)
# delta_at_spot_exfri, max_delta_positive_strike_exfri, max_delta_positive_value_exfri,
# min_delta_negative_strike_exfri, min_delta_negative_value_exfri are from 035f55f4

# Delta Flip from Chart 6 (Delta at Spot Price for Ex-Next Monthly Expiry line, labeled as Delta Flip OI)
chart6_delta_flip_oi_strike = spotPrice # Delta Flip strike is the Spot Price
chart6_delta_flip_oi_value = delta_at_spot_exfri # Delta at Spot Price value

# Max Delta Positivo (Ex-Next Monthly Expiry)
chart6_max_pos_strike = max_delta_positive_strike_exfri
chart6_max_pos_value = max_delta_positive_value_exfri # Already in Millions

# Min Delta Negativo (Ex-Next Monthly Expiry)
chart6_min_neg_strike = min_delta_negative_strike_exfri
chart6_min_neg_value = min_delta_negative_value_exfri # Already in Millions


# ==================== GERAÇÃO DA LINHA ÚNICA (SIMPLIFICADA) ====================

# Criar a string com todos os dados separados por vírgula
# Order: Chart 1, Chart 2, Chart 3, Chart 4, Chart 5, Chart 6, General

# Chart 1 Data (Simplified)
data_string_simple = ""
# Top 5 Neg Gamma (5 strikes * 2 values)
for item in chart1_neg_gamma_data_simple:
    data_string_simple += f"{item['gamma']:.4f},{item['strike']:.0f},"
# Top 5 Pos Gamma (5 strikes * 2 values)
for item in chart1_pos_gamma_data_simple:
     data_string_simple += f"{item['gamma']:.4f},{item['strike']:.0f},"
# Zero Gamma
data_string_simple += f"{chart1_zero_gamma_strike:.0f},"
# OI/Vol Walls (4 walls * 2 values)
data_string_simple += f"{chart1_call_oi_wall_value:.0f},{chart1_call_oi_wall_strike:.0f},"
data_string_simple += f"{chart1_put_oi_wall_value:.0f},{chart1_put_oi_wall_strike:.0f},"
data_string_simple += f"{chart1_call_vol_wall_value:.0f},{chart1_call_vol_wall_strike:.0f},"
data_string_simple += f"{chart1_put_vol_wall_value:.0f},{chart1_put_vol_wall_strike:.0f},"
# Top 5 Calls OI (5 strikes * 2 values)
for item in chart1_top5_call_oi_data:
    data_string_simple += f"{item['StrikePrice']:.0f},{item['CallOpenInt']:.0f},"
# Top 5 Puts OI (5 strikes * 2 values)
for item in chart1_top5_put_oi_data:
    data_string_simple += f"{item['StrikePrice']:.0f},{item['PutOpenInt']:.0f},"
# Largest Volume and Open Interest (4 values * 2 = 8 values)
data_string_simple += f"{chart1_largest_call_vol_strike:.0f},{chart1_largest_call_vol_value:.0f},"
data_string_simple += f"{chart1_largest_put_vol_strike:.0f},{chart1_largest_put_vol_value:.0f},"
data_string_simple += f"{chart1_largest_call_oi_strike:.0f},{chart1_largest_call_oi_value:.0f},"
data_string_simple += f"{chart1_largest_put_oi_strike:.0f},{chart1_largest_put_oi_value:.0f},"


# Chart 2 GEX Levels (5 levels * 3 values)
for item in chart2_gex_levels_data_simple:
    data_string_simple += f"{item['strike']:.0f},{item['call_gex']:.4f},{item['put_gex']:.4f},"

# Chart 3 Gamma Profile points and Delta Flip (Estrutura)
# Gamma Profile points (4 points * 2 values)
data_string_simple += f"{chart3_gamma_flip_value:.4f},{chart3_gamma_flip_strike:.0f},"
data_string_simple += f"{chart3_vol_trigger_value:.4f},{chart3_vol_trigger_strike:.0f},"
data_string_simple += f"{chart3_max_pos_value:.4f},{chart3_max_pos_strike:.0f},"
data_string_simple += f"{chart3_min_neg_value:.4f},{chart3_min_neg_strike:.0f},"
# Delta Flip (Estrutura) from Chart 3
data_string_simple += f"{chart3_delta_flip_estrutura_value:.4f},{chart3_delta_flip_estrutura_strike:.0f},"


# Chart 4 Delta Data (Simplified)
# Top 5 Neg Delta (5 strikes * 2 values)
for item in chart4_neg_delta_data_simple:
    data_string_simple += f"{item['delta']:.4f},{item['strike']:.0f},"
# Top 5 Pos Delta (5 strikes * 2 values) - Changed from 9 to 5 as requested
for item in chart4_pos_delta_data_simple:
    data_string_simple += f"{item['delta']:.4f},{item['strike']:.0f},"
# Zero Delta (based on zero cross) - Not explicitly requested, but including for completeness of Chart 4 context
# data_string_simple += f"{chart4_zero_delta_strike:.0f}," # Removed as not explicitly requested


# Chart 5 DEX Levels (5 levels * 3 values)
for item in chart5_dex_levels_data_simple:
    data_string_simple += f"{item['strike']:.0f},{item['call_dex']:.4f},{item['put_dex']:.4f},"

# Chart 6 Delta Profile points and Delta Flip (OI)
# Delta Flip (OI) from Chart 6 (Delta at Spot Price)
data_string_simple += f"{chart6_delta_flip_oi_value:.4f},{chart6_delta_flip_oi_strike:.0f},"
# Max Delta Positivo (Ex-Next Monthly Expiry)
data_string_simple += f"{chart6_max_pos_value:.4f},{chart6_max_pos_strike:.0f},"
# Min Delta Negativo (Ex-Next Monthly Expiry)
# Add the Delta Flip (Estrutura) strike and value to the data string here - User requested, but already in Chart 3 section. Re-adding here as requested.
data_string_simple += f"{chart3_delta_flip_estrutura_value:.4f},{chart3_delta_flip_estrutura_strike:.0f},"
# Ensure the last value does not have a trailing comma
data_string_simple += f"{chart6_min_neg_value:.4f},{chart6_min_neg_strike:.0f},"


# General Data (3 values + date) - Moved to the end
data_string_simple += f"{spot_price:.2f},{total_gamma:.2f},{df['TotalDelta'].sum():,.2f},{update_date}"


print("📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW:\n")
print("="*80)
print(data_string_simple)
print("="*80 + "\n")

# ==================== INSTRUÇÕES ====================

print("💡 COMO USAR:\n")
print("1. ✅ COPIE a linha acima (entre as linhas ====)")
print("2. ✅ Abra o TradingView e o indicador que você está usando.")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Procure o campo 'Data String' (ou similar) no grupo 'Dados de Entrada'")
print("5. ✅ COLE a linha copiada neste campo")
print("6. ✅ Clique em 'OK'")
print("7. ✅ PRONTO! Os dados serão atualizados automaticamente!")

print("\n" + "="*80)
print("📊 RESUMO DOS DADOS EXTRAÍDOS (SIMPLIFICADO):\n")
print("="*80)

print(f"\n🟢 SPOT PRICE: ${spot_price:,.2f}")
print(f"📊 TOTAL GAMMA: ${total_gamma:.2f} Bn")
print(f"📊 TOTAL DELTA: ${df['TotalDelta'].sum():,.2f} M") # Include Total Delta in summary
print(f"📅 DATA: {update_date}")

print("\n--- CHART 1: GAMMA EXPOSURE ---")
print("\n🔴 Top 5 Strikes por Gamma Exposure Negativo (Ordem Decrescente):")
for item in chart1_neg_gamma_data_simple:
    print(f"  Strike {item['strike']:.0f}: Gamma: {item['gamma']:.4f} Bn")

print("\n🟢 Top 5 Strikes por Gamma Exposure Positivo (Ordem Decrescente):")
for item in chart1_pos_gamma_data_simple:
    print(f"  Strike {item['strike']:.0f}: Gamma: {item['gamma']:.4f} Bn")

print(f"\n🟠 Zero Gamma (Gamma Flip): {chart1_zero_gamma_strike:.0f}")

print("\n📊 Paredes por Open Interest:")
print(f"  Call Wall (OI): {chart1_call_oi_wall_value:.0f} at strike {chart1_call_oi_wall_strike:.0f}")
print(f"  Put Wall (OI): {chart1_put_oi_wall_value:.0f} at strike {chart1_put_oi_wall_strike:.0f}")

print("\n📊 Paredes por Volume:")
print(f"  Call Wall (Vol): {chart1_call_vol_wall_value:.0f} at strike {chart1_call_vol_wall_strike:.0f}")
print(f"  Put Wall (Vol): {chart1_put_vol_wall_value:.0f} at strike {chart1_put_vol_wall_strike:.0f}")

print("\n📈 Top 5 Calls por Open Interest:")
for item in chart1_top5_call_oi_data:
    print(f"  Strike {item['StrikePrice']:.0f}: {item['CallOpenInt']:.0f}")

print("\n📉 Top 5 Puts por Open Interest:")
for item in chart1_top5_put_oi_data:
    print(f"  Strike {item['StrikePrice']:.0f}: {item['PutOpenInt']:.0f}")

print("\n🏆 Strikes com Maior Volume e Open Interest (Geral):")
print(f"  Maior Volume de Call: {chart1_largest_call_vol_value:.0f} at strike {chart1_largest_call_vol_strike:.0f}")
print(f"  Maior Volume de Put: {chart1_largest_put_vol_value:.0f} at strike {chart1_largest_put_vol_strike:.0f}")
print(f"  Maior Open Interest de Call: {chart1_largest_call_oi_value:.0f} at strike {chart1_largest_call_oi_strike:.0f}")
print(f"  Maior Open Interest de Put: {chart1_largest_put_oi_value:.0f} at strike {chart1_largest_put_oi_strike:.0f}")


print("\n--- CHART 2: GEX LEVELS ---")
print("\n💎 Top 5 GEX Levels:")
for i, item in enumerate(chart2_gex_levels_data_simple):
    net_gex = item['call_gex'] + item['put_gex']
    print(f"   {i+1}. Strike {item['strike']:.0f}: Net GEX = {net_gex:.2f} Bn (Call GEX: {item['call_gex']:.4f} Bn, Put GEX: {item['put_gex']:.4f} Bn)")

print("\n--- CHART 3: GAMMA PROFILE POINTS ---")
print("\n🟠 NÍVEIS ESPECIAIS (GAMMA PROFILE):")
print(f"   • Gamma Flip (Ex-Next Monthly): {chart3_gamma_flip_strike:.0f} (γ: {chart3_gamma_flip_value:.4f} Bn)")
print(f"   • Vol Trigger (Gamma Flip Point): {chart3_vol_trigger_strike:.0f} (γ: {chart3_vol_trigger_value:.4f} Bn)")
print(f"   • Max Gamma Pos (Ex-Next Monthly): {chart3_max_pos_strike:.0f} (γ: {chart3_max_pos_value:.4f} Bn)")
print(f"   • Min Gamma Neg (Ex-Next Monthly): {chart3_min_neg_strike:.0f} (γ: {chart3_min_neg_value:.4f} Bn)")
# Print Delta Flip (Estrutura) data using the calculated values
print(f"   • Delta Flip (Estrutura): {chart3_delta_flip_estrutura_strike:.0f} (Δ: {chart3_delta_flip_estrutura_value:.4f} M)")


print("\n--- CHART 4: DELTA EXPOSURE ---")
print("\n🔴 Top 5 Strikes por Delta Exposure Negativo (Ordem Decrescente):")
for item in chart4_neg_delta_data_simple:
    print(f"  Strike {item['strike']:.0f}: Delta: {item['delta']:.4f} M")

print("\n🟢 Top 5 Strikes por Delta Exposure Positivo (Ordem Decrescente):") # Changed from 9 to 5
for item in chart4_pos_delta_data_simple:
    print(f"  Strike {item['strike']:.0f}: Delta: {item['delta']:.4f} M")


print("\n--- CHART 5: DEX LEVELS ---")
print("\n💎 Top 5 DEX Levels:")
for i, item in enumerate(chart5_dex_levels_data_simple):
    net_dex = item['call_dex'] + item['put_dex']
    print(f"   {i+1}. Strike {item['strike']:.0f}: Net DEX = {net_dex:.2f} M (Call DEX: {item['call_dex']:.4f} M, Put DEX: {item['put_dex']:.4f} M)")

print("\n--- CHART 6: DELTA PROFILE POINTS ---")
print("\n🟠 NÍVEIS ESPECIAIS (DELTA PROFILE):")
print(f"   • Delta Flip (OI): {chart6_delta_flip_oi_strike:.0f} (Δ: {chart6_delta_flip_oi_value:.4f} M)")
print(f"   • Max Delta Pos (Ex-Next Monthly): {chart6_max_pos_strike:.0f} (Δ: {chart6_max_pos_value:.4f} M)")
print(f"   • Min Delta Neg (Ex-Next Monthly): {chart6_min_neg_strike:.0f} (Δ: {chart6_min_neg_value:.4f} M)")
# Add the Delta Flip (Estrutura) strike and value to Chart 6 inputs as well
print(f"   • Delta Flip (Estrutura): {chart3_delta_flip_estrutura_strike:.0f} (Δ: {chart3_delta_flip_estrutura_value:.4f} M)")


# Summary of Regimes and Distance
regime_gamma = "POSITIVE GAMMA ✅" if spot_price > chart3_gamma_flip_strike else "NEGATIVE GAMMA ⚠️"
print(f"\n📈 REGIME (GAMMA): {regime_gamma}")

# Check if chart3_gamma_flip_strike is not zero to avoid division by zero
if chart3_gamma_flip_strike != 0:
    distance_to_gamma_flip = ((spot_price - chart3_gamma_flip_strike) / chart3_gamma_flip_strike) * 100
    print(f"📏 DISTÂNCIA DO FLIP (GAMMA): {distance_to_gamma_flip:.2f}%")
else:
    print("📏 DISTÂNCIA DO FLIP (GAMMA): Gamma Flip Strike é zero, não é possível calcular a distância.")


regime_delta = "POSITIVE DELTA ✅" if chart6_delta_flip_oi_value > 0 else "NEGATIVE DELTA ⚠️"
print(f"\n📈 REGIME (DELTA @ SPOT): {regime_delta}")


print(f"\n🕐 Última atualização: {update_date}")

print("\n" + "="*80)
print("✅ DADOS SIMPLIFICADOS PRONTOS PARA USAR!")
print("="*80 + "\n")

# ==================== VALIDAÇÃO (SIMPLIFICADA) ====================

print("🔍 VALIDAÇÃO DOS DADOS:\n")

# Verifica se há dados válidos
errors_simple = []

if spot_price <= 0:
    errors_simple.append("❌ Spot Price inválido")

# Basic checks for Chart 1 Gamma data (Simplified)
if not chart1_neg_gamma_data_simple:
    errors_simple.append("❌ Dados de Top 5 Gamma Negativo (Chart 1) não coletados")
if not chart1_pos_gamma_data_simple:
    errors_simple.append("❌ Dados de Top 5 Gamma Positivo (Chart 1) não coletados")
if chart1_zero_gamma_strike <= 0:
    errors_simple.append("❌ Zero Gamma (Chart 1) inválido")
if chart1_call_oi_wall_strike <= 0 or chart1_put_oi_wall_strike <= 0 or chart1_call_vol_wall_strike <= 0 or chart1_put_vol_wall_strike <= 0:
     errors_simple.append("❌ Strikes de Paredes OI/Volume (Chart 1) inválidos")


# Basic checks for Chart 2 GEX Levels (Simplified)
if not chart2_gex_levels_data_simple:
    errors_simple.append("❌ Dados do Chart 2 (GEX Levels - Top 5) não coletados")

# Basic checks for Chart 3 Gamma Profile
if chart3_gamma_flip_strike <= 0:
    errors_simple.append("❌ Gamma Flip Strike (Chart 3) inválido")
if chart3_vol_trigger_strike <= 0:
     errors_simple.append("❌ Vol Trigger Strike (Chart 3) inválido")
if chart3_max_pos_strike <= 0 or chart3_min_neg_strike <= 0:
     errors_simple.append("❌ Strikes de Max/Min Gamma (Chart 3) inválidos")
# Check for Chart 3 Delta Flip (Estrutura)
if chart3_delta_flip_estrutura_strike <= 0:
     errors_simple.append("❌ Delta Flip (Estrutura) Strike (Chart 3) inválido")


# Basic checks for Chart 4 Delta Exposure (Simplified)
if not chart4_neg_delta_data_simple:
    errors_simple.append("❌ Dados de Top 5 Delta Negativo (Chart 4) não coletados")
if not chart4_pos_delta_data_simple:
    errors_simple.append("❌ Dados de Top 5 Delta Positivo (Chart 4) não coletados")


# Basic checks for Chart 5 DEX Levels (Simplified)
if not chart5_dex_levels_data_simple:
    errors_simple.append("❌ Dados do Chart 5 (DEX Levels - Top 5) não coletados")

# Basic checks for Chart 6 Delta Profile
if chart6_delta_flip_oi_strike <= 0: # This is spot price, already checked
     errors_simple.append("❌ Delta Flip OI Strike (Chart 6) inválido")
if chart6_max_pos_strike <= 0 or chart6_min_neg_strike <= 0:
     errors_simple.append("❌ Strikes de Max/Min Delta (Chart 6) inválidos")


if errors_simple:
    print("⚠️ AVISOS ENCONTRADOS:")
    for error in errors_simple:
        print(f"   {error}")
else:
    print("✅ Todos os dados validados com sucesso!")
    print("✅ Pronto para usar no TradingView!")

print("\n" + "="*80 + "\n")


🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA/DELTA FLIP (VERSÃO SIMPLIFICADA)

📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW:

-1.0533,6400,-1.1908,6300,-1.4016,6500,-1.5057,6895,-2.7925,6890,10.9452,7000,5.8570,6950,4.4717,6850,3.9613,7100,3.7872,6900,6773,363849,5000,399274,5000,85424,6900,73442,6850,5000,363849,6000,241557,4000,200147,5000,192934,6000,190980,5000,399274,6000,264839,4000,225537,6000,206531,5000,205096,6900,85424,6850,73442,5000,363849,5000,399274,7000,16.6733,-5.7281,6000,8.1595,-9.1982,6900,10.4918,-6.7047,6800,7.6095,-5.3291,6700,5.8691,-6.1893,12.1592,6821,-3.0775,6773,64.8422,7007,-65.0810,5886,2582143.1035,6891,-406.3461,2700,-500.8333,6290,-606.2408,5340,-1032.8980,12000,-3696.6134,6330,664797.8352,5000,497895.0896,6000,244326.1315,4000,55834.8764,6600,52337.5785,6700,5000,697814.7929,-33016.9577,6000,580675.1381,-82780.0485,4000,251606.5871,-7280.4556,7000,90959.6276,-45971.6997,6700,85054.2724,-32716.6939,2582143.1035,6891,3098571.7242,8269,2582143.1035,

### 5 DTE

In [131]:
# Consolidating results for 5 DTE - GAMMA

if df_5dte.empty:
    print("Cannot provide consolidated results for 5 DTE as no options were found.")
else:
    print("📊 RESUMO CONSOLIDADO DOS DADOS (5 DTE)")
    print("="*80)

    # --- DADOS DO CHART 1 (Gamma Exposure) para 5 DTE ---
    print("\n--- DADOS DO CHART 1 (Gamma Exposure) para 5 DTE ---")

    # Check if dfAgg_sorted_5dte exists (calculated in cell 0981fa28)
    if 'dfAgg_sorted_5dte' in locals():
        # Get the top 9 smallest gamma values (most negative) for 5 DTE
        smallest_gamma_5dte_summary = dfAgg_sorted_5dte.head(9)
        print("\nTop 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente, 5 DTE):")
        for index, row in smallest_gamma_5dte_summary.iloc[::-1].iterrows():
             # Get the corresponding Delta value from dfAgg_delta_5dte if it exists
            delta_value_5dte_summary = dfAgg_delta_5dte.loc[index, 'TotalDelta'] if 'dfAgg_delta_5dte' in locals() and index in dfAgg_delta_5dte.index else 0
            print(f"  Strike {index:.0f}: Type: P, Delta: {delta_value_5dte_summary:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

        # Get the top 9 largest gamma values (most positive) for 5 DTE
        largest_gamma_5dte_summary = dfAgg_sorted_5dte.tail(9)
        print("\nTop 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente, 5 DTE):")
        for index, row in largest_gamma_5dte_summary.iloc[::-1].iterrows():
             # Get the corresponding Delta value from dfAgg_delta_5dte if it exists
            delta_value_5dte_summary = dfAgg_delta_5dte.loc[index, 'TotalDelta'] if 'dfAgg_delta_5dte' in locals() and index in dfAgg_delta_5dte.index else 0
            print(f"  Strike {index:.0f}: Type: C, Delta: {delta_value_5dte_summary:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

        # Add the Zero Gamma (Gamma Flip) point for 5 DTE
        # Check if zeroGamma_5dte exists (calculated in cell 0981fa28)
        if 'zeroGamma_5dte' in locals() and zeroGamma_5dte is not None:
            print(f"\nZero Gamma (Gamma Flip, 5 DTE): {zeroGamma_5dte:.0f}")
        else:
            print("\nZero Gamma (Gamma Flip, 5 DTE): Não encontrado")

        # Print the total gamma for 5 DTE
        print(f"\nTotal Gamma (5 DTE): ${df_5dte['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")
    else:
        print("\nGamma Exposure data for 5 DTE not available. Please run the relevant cells.")


    # --- Dados de Open Interest e Volume para 5 DTE ---
    print("\n--- Dados de Open Interest e Volume para 5 DTE ---")

    # Check if variables from cell 04f5652b exist
    if 'call_oi_wall_strike_5dte' in locals():
        print("\nParedes por Open Interest (5 DTE):")
        print(f"  Call Wall (OI): {call_oi_wall_value_5dte:.0f} at strike {call_oi_wall_strike_5dte:.0f}")
        print(f"  Put Wall (OI): {put_oi_wall_value_5dte:.0f} at strike {put_oi_wall_strike_5dte:.0f}")

        print("\nParedes por Volume (5 DTE):")
        print(f"  Call Wall (Vol): {call_vol_wall_value_5dte:.0f} at strike {call_vol_wall_strike_5dte:.0f}")
        print(f"  Put Wall (Vol): {put_vol_wall_value_5dte:.0f} at strike {put_vol_wall_strike_5dte:.0f}")

        # Check if top5_call_oi_5dte and top5_put_oi_5dte exist
        if 'top5_call_oi_5dte' in locals():
            print("\nTop 5 Calls por Open Interest (5 DTE):")
            for index, row in top5_call_oi_5dte.iterrows():
                print(f"  Strike {row['StrikePrice']:.0f}: {row['CallOpenInt']:.0f}")
        else:
            print("\nTop 5 Calls por Open Interest (5 DTE) data not available.")

        if 'top5_put_oi_5dte' in locals():
            print("\nTop 5 Puts por Open Interest (5 DTE):")
            for index, row in top5_put_oi_5dte.iterrows():
                print(f"  Strike {row['StrikePrice']:.0f}: {row['PutOpenInt']:.0f}")
        else:
             print("\nTop 5 Puts por Open Interest (5 DTE) data not available.")


        print("\n--- Strikes com Maior Volume e Open Interest (Geral, 5 DTE) ---")
        # Check if largest_call_vol_strike_5dte etc. exist
        if 'largest_call_vol_strike_5dte' in locals():
            print(f"\nMaior Volume de Call: {largest_call_vol_strike_5dte['CallVol']:.0f} at strike {largest_call_vol_strike_5dte['StrikePrice']:.0f}")
        else:
            print("\nMaior Volume de Call (5 DTE) data not available.")

        if 'largest_put_vol_strike_5dte' in locals():
            print(f"Maior Volume de Put: {largest_put_vol_strike_5dte['PutVol']:.0f} at strike {largest_put_vol_strike_5dte['StrikePrice']:.0f}")
        else:
            print("Maior Volume de Put (5 DTE) data not available.")

        if 'largest_call_oi_strike_5dte' in locals():
            print(f"\nMaior Open Interest de Call: {largest_call_oi_strike_5dte['CallOpenInt']:.0f} at strike {largest_call_oi_strike_5dte['StrikePrice']:.0f}")
        else:
            print("\nMaior Open Interest de Call (5 DTE) data not available.")

        if 'largest_put_oi_strike_5dte' in locals():
            print(f"Maior Open Interest de Put: {largest_put_oi_strike_5dte['PutOpenInt']:.0f} at strike {largest_put_oi_strike_5dte['StrikePrice']:.0f}")
        else:
            print("Maior Open Interest de Put (5 DTE) data not available.")

    else:
        print("\nOpen Interest and Volume data for 5 DTE not available. Please run the relevant cells.")


    # --- DADOS CHART 2 (GEX Levels) para 5 DTE ---
    print("\n--- DADOS CHART 2 (GEX Levels) para 5 DTE ---")
    # Check if gex_levels_5dte exists (calculated in cell 05caf7df)
    if 'gex_levels_5dte' in locals() and not gex_levels_5dte.empty:
        print("\nTop 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure, 5 DTE):")
        for i in range(len(gex_levels_5dte)):
            strike = gex_levels_5dte.index[i]
            call_gex = gex_levels_5dte.iloc[i]['CallGEX'] / 10**9 # Convert to billions
            put_gex = gex_levels_5dte.iloc[i]['PutGEX'] / 10**9   # Convert to billions
            print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")
    else:
        print("\nChart 2 (GEX Levels) data for 5 DTE not available or empty. Please run the relevant cells.")


    # --- DADOS CHART 3 (Gamma Profile Points) para 5 DTE ---
    print("\n--- DADOS CHART 3 (Gamma Profile Points) para 5 DTE ---")
    # Check if variables from cell 0b7b3243 exist
    if 'zeroGamma_5dte_profile' in locals():
        if zeroGamma_5dte_profile is not None:
            print(f"\nGamma Flip (5 DTE Profile): {zeroGamma_5dte_profile:.0f}")
        else:
            print("\nGamma Flip (5 DTE Profile): Não encontrado")

        if 'vol_trigger_value_at_flip_5dte' in locals():
            print(f"Vol Trigger (Gamma Flip Point, 5 DTE Profile): {vol_trigger_value_at_flip_5dte:.4f} at strike {zeroGamma_5dte_profile:.0f}")
        else:
             print("Vol Trigger (5 DTE Profile) data not available.")


        if 'max_gamma_positive_strike_5dte' in locals():
            print(f"Max Gamma Positivo (5 DTE Profile): {max_gamma_positive_value_5dte:.4f} at strike {max_gamma_positive_strike_5dte:.0f}")
        else:
             print("Max Gamma Positivo (5 DTE Profile) data not available.")

        if 'min_gamma_negative_strike_5dte' in locals():
            print(f"Min Gamma Negativo (5 DTE Profile): {min_gamma_negative_value_5dte:.4f} at strike {min_gamma_negative_strike_5dte:.0f}")
        else:
             print("Min Gamma Negativo (5 DTE Profile) data not available.")
    else:
        print("\nChart 3 (Gamma Profile Points) data for 5 DTE not available. Please run the relevant cells.")


    print("\n" + "="*80)
    print("✅ RESUMO CONSOLIDADO PARA 5 DTE GERADO COM SUCESSO!")
    print("="*80 + "\n")

NameError: name 'df_5dte' is not defined

In [ ]:
# Consolidating Chart 4 and Chart 5 results for 5 DTE - DELTA

if df_5dte.empty:
    print("Cannot provide consolidated Chart 4 and 5 results for 5 DTE as no options were found.")
else:
    print("📊 RESUMO CONSOLIDADO DOS DADOS (Chart 4 and Chart 5) para 5 DTE")
    print("="*80)

    # --- DADOS CHART 4 para 5 DTE ---
    print("\n--- DADOS CHART 4 para 5 DTE ---")

    # Check if dfAgg_delta_5dte exists (calculated in cell 0981fa28)
    if 'dfAgg_delta_5dte' in locals():
        dfAgg_delta_sorted_5dte = dfAgg_delta_5dte.sort_values(by='TotalDelta')

        # Get the top 9 smallest delta exposure values (most negative) for 5 DTE
        smallest_delta_exposure_5dte = dfAgg_delta_sorted_5dte.head(9)
        print("\nTop 9 Strikes por Delta Exposure Negativo (Ordem Decrescente, 5 DTE):")
        for index, row in smallest_delta_exposure_5dte.iloc[::-1].iterrows():
            # Need gamma value from dfAgg_5dte if available (calculated in 0981fa28)
            gamma_value_5dte_summary = dfAgg_5dte.loc[index, 'TotalGamma'] if 'dfAgg_5dte' in locals() and index in dfAgg_5dte.index else 0
            print(f"  Strike {index:.0f}: Type: P, Gamma: {gamma_value_5dte_summary:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")

        # Get the top 9 largest delta exposure values (most positive) for 5 DTE
        largest_delta_exposure_5dte = dfAgg_delta_sorted_5dte.tail(9)
        print("\nTop 9 Strikes por Delta Exposure Positivo (Ordem Decrescente, 5 DTE):")
        for index, row in largest_delta_exposure_5dte.iloc[::-1].iterrows():
             # Need gamma value from dfAgg_5dte if available
            gamma_value_5dte_summary = dfAgg_5dte.loc[index, 'TotalGamma'] if 'dfAgg_5dte' in locals() and index in dfAgg_5dte.index else 0
            print(f"  Strike {index:.0f}: Type: C, Gamma: {gamma_value_5dte_summary:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")

        # Add the Zero Delta (Delta Flip) point for 5 DTE
        # Note: We haven't calculated a separate Delta Flip for the 5 DTE profile
        # in the same way as the main one. Reusing the logic from the 5 DTE Chart 1 summary
        # which looks for a zero cross in the aggregated Delta.
        zeroCrossIdx_delta_5dte = np.where(np.diff(np.sign(dfAgg_delta_sorted_5dte['TotalDelta'])))[0]

        if zeroCrossIdx_delta_5dte.size > 0:
             # Find the two closest strikes to the *first* zero cross
            first_zero_cross_idx_delta_5dte = zeroCrossIdx_delta_5dte[0]
            negDelta_5dte_summary = dfAgg_delta_sorted_5dte['TotalDelta'].iloc[first_zero_cross_idx_delta_5dte]
            posDelta_5dte_summary = dfAgg_delta_sorted_5dte['TotalDelta'].iloc[first_zero_cross_idx_delta_5dte+1]
            negStrike_delta_5dte_summary = dfAgg_delta_sorted_5dte.index[first_zero_cross_idx_delta_5dte]
            posStrike_delta_5dte_summary = dfAgg_delta_sorted_5dte.index[first_zero_cross_idx_delta_5dte+1]

            # Linear interpolation to find the zero delta strike
            zeroDelta_5dte_summary = posStrike_delta_5dte_summary - ((posStrike_delta_5dte_summary - negStrike_delta_5dte_summary) * posDelta_5dte_summary/(posDelta_5dte_summary-negDelta_5dte_summary))
            print(f"\nZero Delta (Delta Flip - Zero Cross, 5 DTE): {zeroDelta_5dte_summary:.0f}")
        else:
            print("\nZero Delta (Delta Flip - Zero Cross, 5 DTE): Não encontrado")


        # Print the total delta exposure for 5 DTE
        print(f"\nTotal Delta (5 DTE): ${df_5dte['TotalDelta'].sum():,.2f} Million per 1% SPX Move")

        # --- Strikes com Maior Volume e Open Interest (Geral, 5 DTE) ---
        # These were already calculated in cell 04f5652b, reuse variables
        print("\n--- Strikes com Maior Volume e Open Interest (Geral, 5 DTE) ---")
        if 'largest_call_vol_strike_5dte' in locals():
            print(f"\nMaior Volume de Call: {largest_call_vol_strike_5dte['CallVol']:.0f} at strike {largest_call_vol_strike_5dte['StrikePrice']:.0f}")
        else:
            print("\nMaior Volume de Call (5 DTE) data not available.")

        if 'largest_put_vol_strike_5dte' in locals():
            print(f"Maior Volume de Put: {largest_put_vol_strike_5dte['PutVol']:.0f} at strike {largest_put_vol_strike_5dte['StrikePrice']:.0f}")
        else:
            print("Maior Volume de Put (5 DTE) data not available.")

        if 'largest_call_oi_strike_5dte' in locals():
            print(f"\nMaior Open Interest de Call: {largest_call_oi_strike_5dte['CallOpenInt']:.0f} at strike {largest_call_oi_strike_5dte['StrikePrice']:.0f}")
        else:
            print("\nMaior Open Interest de Call (5 DTE) data not available.")

        if 'largest_put_oi_strike_5dte' in locals():
            print(f"Maior Open Interest de Put: {largest_put_oi_strike_5dte['PutOpenInt']:.0f} at strike {largest_put_oi_strike_5dte['StrikePrice']:.0f}")
        else:
            print("Maior Open Interest de Put (5 DTE) data not available.")

    else:
        print("\nChart 4 data for 5 DTE not available. Please run the relevant cells.")


    # --- DADOS CHART 5 para 5 DTE ---
    print("\n--- DADOS CHART 5 para 5 DTE ---")
    # Check if dfAgg_delta_5dte exists and has 'AbsoluteTotalDEX' (calculated in 05caf7df)
    if 'dfAgg_delta_5dte' in locals() and 'AbsoluteTotalDEX' in dfAgg_delta_5dte.columns:
        dfAgg_delta_sorted_dex_5dte = dfAgg_delta_5dte.sort_values(by='AbsoluteTotalDEX', ascending=False)
        dex_levels_5dte_summary = dfAgg_delta_sorted_dex_5dte.head(6)

        print("\nTop 6 DEX Levels (based on sum of absolute Call and Put Delta Exposure, 5 DTE):")
        for i in range(len(dex_levels_5dte_summary)):
            strike = dex_levels_5dte_summary.index[i]
            call_dex = dex_levels_5dte_summary.iloc[i]['CallDEX'] / 10**6 # Convert to millions
            put_dex = dex_levels_5dte_summary.iloc[i]['PutDEX'] / 10**6   # Convert to millions
            print(f"  DEX Level {i+1}: Strike {strike:.0f}, Call DEX: {call_dex:.4f} Million, Put DEX: {put_dex:.4f} Million")
    else:
        print("\nChart 5 data for 5 DTE not available. Please run the relevant cells.")


    print("\n" + "="*80)
    print("✅ RESUMO CONSOLIDADO (Chart 4 and Chart 5) PARA 5 DTE GERADO COM SUCESSO!")
    print("="*80 + "\n")

### 0 DTE

In [ ]:
# Consolidating results for 0 DTE - GAMMA

if df_0dte.empty:
    print("Cannot provide consolidated results for 0 DTE as no options were found.")
else:
    print("📊 RESUMO CONSOLIDADO DOS DADOS (0 DTE)")
    print("="*80)

    # --- DADOS DO CHART 1 (Gamma Exposure) para 0 DTE ---
    print("\n--- DADOS DO CHART 1 (Gamma Exposure) para 0 DTE ---")

    # Check if dfAgg_sorted_0dte exists (calculated in cell f8da9cbc)
    if 'dfAgg_sorted_0dte' in locals():
        # Get the top 9 smallest gamma values (most negative) for 0 DTE
        smallest_gamma_0dte_summary = dfAgg_sorted_0dte.head(9)
        print("\nTop 9 Strikes por Gamma Exposure Negativo (Ordem Decrescente, 0 DTE):")
        for index, row in smallest_gamma_0dte_summary.iloc[::-1].iterrows():
             # Get the corresponding Delta value from dfAgg_delta_0dte if it exists
            delta_value_0dte_summary = dfAgg_delta_0dte.loc[index, 'TotalDelta'] if 'dfAgg_delta_0dte' in locals() and index in dfAgg_delta_0dte.index else 0
            print(f"  Strike {index:.0f}: Type: P, Delta: {delta_value_0dte_summary:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

        # Get the top 9 largest gamma values (most positive) for 0 DTE
        largest_gamma_0dte_summary = dfAgg_sorted_0dte.tail(9)
        print("\nTop 9 Strikes por Gamma Exposure Positivo (Ordem Decrescente, 0 DTE):")
        for index, row in largest_gamma_0dte_summary.iloc[::-1].iterrows():
             # Get the corresponding Delta value from dfAgg_delta_0dte if it exists
            delta_value_0dte_summary = dfAgg_delta_0dte.loc[index, 'TotalDelta'] if 'dfAgg_delta_0dte' in locals() and index in dfAgg_delta_0dte.index else 0
            print(f"  Strike {index:.0f}: Type: C, Delta: {delta_value_0dte_summary:.4f} M, Gamma: {row['TotalGamma']:.4f} Bn")

        # Add the Zero Gamma (Gamma Flip) point for 0 DTE
        # Check if zeroGamma_0dte exists (calculated in cell f8da9cbc)
        if 'zeroGamma_0dte' in locals() and zeroGamma_0dte is not None:
            print(f"\nZero Gamma (Gamma Flip, 0 DTE): {zeroGamma_0dte:.0f}")
        else:
            print("\nZero Gamma (Gamma Flip, 0 DTE): Não encontrado")

        # Print the total gamma for 0 DTE
        print(f"\nTotal Gamma (0 DTE): ${df_0dte['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")
    else:
        print("\nGamma Exposure data for 0 DTE not available. Please run the relevant cells.")


    # --- Dados de Open Interest e Volume para 0 DTE ---
    print("\n--- Dados de Open Interest e Volume para 0 DTE ---")

    # Check if variables from cell f8904fb2 exist
    if 'call_oi_wall_strike_0dte' in locals():
        print("\nParedes por Open Interest (0 DTE):")
        print(f"  Call Wall (OI): {call_oi_wall_value_0dte:.0f} at strike {call_oi_wall_strike_0dte:.0f}")
        print(f"  Put Wall (OI): {put_oi_wall_value_0dte:.0f} at strike {put_oi_wall_strike_0dte:.0f}")

        print("\nParedes por Volume (0 DTE):")
        print(f"  Call Wall (Vol): {call_vol_wall_value_0dte:.0f} at strike {call_vol_wall_strike_0dte:.0f}")
        print(f"  Put Wall (Vol): {put_vol_wall_value_0dte:.0f} at strike {put_vol_wall_strike_0dte:.0f}")

        # Check if top5_call_oi_0dte and top5_put_oi_0dte exist
        if 'top5_call_oi_0dte' in locals():
            print("\nTop 5 Calls por Open Interest (0 DTE):")
            for index, row in top5_call_oi_0dte.iterrows():
                print(f"  Strike {row['StrikePrice']:.0f}: {row['CallOpenInt']:.0f}")
        else:
            print("\nTop 5 Calls por Open Interest (0 DTE) data not available.")

        if 'top5_put_oi_0dte' in locals():
            print("\nTop 5 Puts por Open Interest (0 DTE):")
            for index, row in top5_put_oi_0dte.iterrows():
                print(f"  Strike {row['StrikePrice']:.0f}: {row['PutOpenInt']:.0f}")
        else:
             print("\nTop 5 Puts por Open Interest (0 DTE) data not available.")


        print("\n--- Strikes com Maior Volume e Open Interest (Geral, 0 DTE) ---")
        # Check if largest_call_vol_strike_0dte etc. exist
        if 'largest_call_vol_strike_0dte' in locals():
            print(f"\nMaior Volume de Call: {largest_call_vol_strike_0dte['CallVol']:.0f} at strike {largest_call_vol_strike_0dte['StrikePrice']:.0f}")
        else:
            print("\nMaior Volume de Call (0 DTE) data not available.")

        if 'largest_put_vol_strike_0dte' in locals():
            print(f"Maior Volume de Put: {largest_put_vol_strike_0dte['PutVol']:.0f} at strike {largest_put_vol_strike_0dte['StrikePrice']:.0f}")
        else:
            print("Maior Volume de Put (0 DTE) data not available.")

        if 'largest_call_oi_strike_0dte' in locals():
            print(f"\nMaior Open Interest de Call: {largest_call_oi_strike_0dte['CallOpenInt']:.0f} at strike {largest_call_oi_strike_0dte['StrikePrice']:.0f}")
        else:
            print("\nMaior Open Interest de Call (0 DTE) data not available.")

        if 'largest_put_oi_strike_0dte' in locals():
            print(f"Maior Open Interest de Put: {largest_put_oi_strike_0dte['PutOpenInt']:.0f} at strike {largest_put_oi_strike_0dte['StrikePrice']:.0f}")
        else:
            print("Maior Open Interest de Put (0 DTE) data not available.")

    else:
        print("\nOpen Interest and Volume data for 0 DTE not available. Please run the relevant cells.")


    # --- DADOS CHART 2 (GEX Levels) para 0 DTE ---
    print("\n--- DADOS CHART 2 (GEX Levels) para 0 DTE ---")
    # Check if gex_levels_0dte exists (calculated in cell 12d29a5a)
    if 'gex_levels_0dte' in locals() and not gex_levels_0dte.empty:
        print("\nTop 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure, 0 DTE):")
        for i in range(len(gex_levels_0dte)):
            strike = gex_levels_0dte.index[i]
            call_gex = gex_levels_0dte.iloc[i]['CallGEX'] / 10**9 # Convert to billions
            put_gex = gex_levels_0dte.iloc[i]['PutGEX'] / 10**9   # Convert to billions
            print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")
    else:
        print("\nChart 2 (GEX Levels) data for 0 DTE not available or empty. Please run the relevant cells.")


    # --- DADOS CHART 3 (Gamma Profile Points) para 0 DTE ---
    print("\n--- DADOS CHART 3 (Gamma Profile Points) para 0 DTE ---")
    # Check if variables from cell b0d56d49 exist
    if 'zeroGamma_0dte_profile' in locals():
        if zeroGamma_0dte_profile is not None:
            print(f"\nGamma Flip (0 DTE Profile): {zeroGamma_0dte_profile:.0f}")
        else:
            print("\nGamma Flip (0 DTE Profile): Não encontrado")

        if 'vol_trigger_value_at_flip_0dte' in locals():
            print(f"Vol Trigger (Gamma Flip Point, 0 DTE Profile): {vol_trigger_value_at_flip_0dte:.4f} at strike {zeroGamma_0dte_profile:.0f}")
        else:
             print("Vol Trigger (0 DTE Profile) data not available.")


        if 'max_gamma_positive_strike_0dte' in locals():
            print(f"Max Gamma Positivo (0 DTE Profile): {max_gamma_positive_value_0dte:.4f} at strike {max_gamma_positive_strike_0dte:.0f}")
        else:
             print("Max Gamma Positivo (0 DTE Profile) data not available.")

        if 'min_gamma_negative_strike_0dte' in locals():
            print(f"Min Gamma Negativo (0 DTE Profile): {min_gamma_negative_value_0dte:.4f} at strike {min_gamma_negative_strike_0dte:.0f}")
        else:
             print("Min Gamma Negativo (0 DTE Profile) data not available.")
    else:
        print("\nChart 3 (Gamma Profile Points) data for 0 DTE not available. Please run the relevant cells.")


    print("\n" + "="*80)
    print("✅ RESUMO CONSOLIDADO PARA 0 DTE GERADO COM SUCESSO!")
    print("="*80 + "\n")

In [ ]:
# Consolidating Chart 4 and Chart 5 results for 0 DTE - DELTA

if df_0dte.empty:
    print("Cannot provide consolidated Chart 4 and 5 results for 0 DTE as no options were found.")
else:
    print("📊 RESUMO CONSOLIDADO DOS DADOS (Chart 4 and Chart 5) para 0 DTE")
    print("="*80)

    # --- DADOS CHART 4 para 0 DTE ---
    print("\n--- DADOS CHART 4 para 0 DTE ---")

    # Check if dfAgg_delta_0dte exists (calculated in cell f8da9cbc)
    if 'dfAgg_delta_0dte' in locals():
        dfAgg_delta_sorted_0dte = dfAgg_delta_0dte.sort_values(by='TotalDelta')

        # Get the top 9 smallest delta exposure values (most negative) for 0 DTE
        smallest_delta_exposure_0dte = dfAgg_delta_sorted_0dte.head(9)
        print("\nTop 9 Strikes por Delta Exposure Negativo (Ordem Decrescente, 0 DTE):")
        for index, row in smallest_delta_exposure_0dte.iloc[::-1].iterrows():
             # Need gamma value from dfAgg_0dte if available (calculated in f8da9cbc)
            gamma_value_0dte_summary = dfAgg_0dte.loc[index, 'TotalGamma'] if 'dfAgg_0dte' in locals() and index in dfAgg_0dte.index else 0
            print(f"  Strike {index:.0f}: Type: P, Gamma: {gamma_value_0dte_summary:.4f} Bn, Delta: {row['TotalDelta']:.4f} M")

        # Get the top 9 largest delta exposure values (most positive) for 0 DTE
        largest_delta_exposure_0dte = dfAgg_delta_sorted_0dte.tail(9)
        print("\nTop 9 Strikes por Delta Exposure Positivo (Ordem Decrescente, 0 DTE):")
        for index, row in largest_delta_exposure_0dte.iloc[::-1].iterrows():
             # Need gamma value from dfAgg_0dte if available
            gamma_value_0dte_summary = dfAgg_0dte.loc[index, 'TotalGamma'] if 'dfAgg_0dte' in locals() and index in dfAgg_0dte.index else 0
            print(f"  Strike {index:.0f}: Type: C, Delta: {delta_value_0dte_summary:.4f} M, Gamma: {gamma_value_0dte_summary:.4f} Bn")

        # Add the Zero Delta (Delta Flip) point for 0 DTE
        # Reusing the logic from the 0 DTE Chart 1 summary
        zeroCrossIdx_delta_0dte = np.where(np.diff(np.sign(dfAgg_delta_sorted_0dte['TotalDelta'])))[0]

        if zeroCrossIdx_delta_0dte.size > 0:
             # Find the two closest strikes to the *first* zero cross
            first_zero_cross_idx_delta_0dte = zeroCrossIdx_delta_0dte[0]
            negDelta_0dte_summary = dfAgg_delta_sorted_0dte['TotalDelta'].iloc[first_zero_cross_idx_delta_0dte]
            posDelta_0dte_summary = dfAgg_delta_sorted_0dte['TotalDelta'].iloc[first_zero_cross_idx_delta_0dte+1]
            negStrike_delta_0dte_summary = dfAgg_delta_sorted_0dte.index[first_zero_cross_idx_delta_0dte]
            posStrike_delta_0dte_summary = dfAgg_delta_sorted_0dte.index[first_zero_cross_idx_delta_0dte+1]

            # Linear interpolation to find the zero delta strike
            zeroDelta_0dte_summary = posStrike_delta_0dte_summary - ((posStrike_delta_0dte_summary - negStrike_delta_0dte_summary) * posDelta_0dte_summary/(posDelta_0dte_summary-negDelta_0dte_summary))
            print(f"\nZero Delta (Delta Flip - Zero Cross, 0 DTE): {zeroDelta_0dte_summary:.0f}")
        else:
            print("\nZero Delta (Delta Flip - Zero Cross, 0 DTE): Não encontrado")


        # Print the total delta exposure for 0 DTE
        print(f"\nTotal Delta (0 DTE): ${df_0dte['TotalDelta'].sum():,.2f} Million per 1% SPX Move")

        # --- Strikes com Maior Volume e Open Interest (Geral, 0 DTE) ---
        # These were already calculated in cell f8904fb2, reuse variables
        print("\n--- Strikes com Maior Volume e Open Interest (Geral, 0 DTE) ---")
        if 'largest_call_vol_strike_0dte' in locals():
            print(f"\nMaior Volume de Call: {largest_call_vol_strike_0dte['CallVol']:.0f} at strike {largest_call_vol_strike_0dte['StrikePrice']:.0f}")
        else:
            print("\nMaior Volume de Call (0 DTE) data not available.")

        if 'largest_put_vol_strike_0dte' in locals():
            print(f"Maior Volume de Put: {largest_put_vol_strike_0dte['PutVol']:.0f} at strike {largest_put_vol_strike_0dte['StrikePrice']:.0f}")
        else:
            print("Maior Volume de Put (0 DTE) data not available.")

        if 'largest_call_oi_strike_0dte' in locals():
            print(f"\nMaior Open Interest de Call: {largest_call_oi_strike_0dte['CallOpenInt']:.0f} at strike {largest_call_oi_strike_0dte['StrikePrice']:.0f}")
        else:
            print("\nMaior Open Interest de Call (0 DTE) data not available.")

        if 'largest_put_oi_strike_0dte' in locals():
            print(f"Maior Open Interest de Put: {largest_put_oi_strike_0dte['PutOpenInt']:.0f} at strike {largest_put_oi_strike_0dte['StrikePrice']:.0f}")
        else:
            print("Maior Open Interest de Put (0 DTE) data not available.")

    else:
        print("\nChart 4 data for 0 DTE not available. Please run the relevant cells.")


    # --- DADOS CHART 5 para 0 DTE ---
    print("\n--- DADOS CHART 5 para 0 DTE ---")
    # Check if dfAgg_delta_0dte exists and has 'AbsoluteTotalDEX' (calculated in 12d29a5a)
    if 'dfAgg_delta_0dte' in locals() and 'AbsoluteTotalDEX' in dfAgg_delta_0dte.columns:
        dfAgg_delta_sorted_dex_0dte = dfAgg_delta_0dte.sort_values(by='AbsoluteTotalDEX', ascending=False)
        dex_levels_0dte_summary = dfAgg_delta_sorted_dex_0dte.head(6)

        print("\nTop 6 DEX Levels (based on sum of absolute Call and Put Delta Exposure, 0 DTE):")
        for i in range(len(dex_levels_0dte_summary)):
            strike = dex_levels_0dte_summary.index[i]
            call_dex = dex_levels_0dte_summary.iloc[i]['CallDEX'] / 10**6 # Convert to millions
            put_dex = dex_levels_0dte_summary.iloc[i]['PutDEX'] / 10**6   # Convert to millions
            print(f"  DEX Level {i+1}: Strike {strike:.0f}, Call DEX: {call_dex:.4f} Million, Put DEX: {put_dex:.4f} Million")
    else:
        print("\nChart 5 data for 0 DTE not available. Please run the relevant cells.")


    print("\n" + "="*80)
    print("✅ RESUMO CONSOLIDADO (Chart 4 and Chart 5) PARA 0 DTE GERADO COM SUCESSO!")
    print("="*80 + "\n")